# Paper results: BM25 text and image retrieval

Text compares the BM25 baseline, NV-Retriever, GOLD-mined, and GOLD-reweighted models. The baseline selects a different member of the top-five BM25 set; GOLD-mined retains the query-construction negative; GOLD-reweighted adds graded supervision. Images use two different same-category negatives. NV-Retriever mines from each modality's baseline training dataset.

All four strategies share held-out queries, corpus, and comparison pairs within each modality. The analysis verifies these identities before plotting or computing paired comparisons. Graded InfoNCE uses v18 in place of v3, with V=20 selected on validation for both modalities. All main results use seeds 42, 43, and 44. Text reports Recall@5; images report Recall@20.

Run All refreshes human data and evaluations, checks prediction provenance, and rebuilds the figures and tables. Sources are configured in `analysis/active_training.json`.


In [ ]:
# @claude, when editing the notebook, please actually run it
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import pandas as pd
import seaborn as sns

from importlib import reload
import utils.training_profile as profile_module
reload(profile_module)
import utils.paper_analysis as analysis_module
reload(analysis_module)

from utils.paper_analysis import discover_runs as discover_all_runs, discover_profile_runs, health_check, active_dataset_bases
from utils.training_profile import training_profile, analysis_profiles
from utils.training_plan import RETIRED_INFONCE_STYLES

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

ANALYSIS_QUERY_KIND = "rephrased"
ANALYSIS_REPHRASE = "in-context"
BASELINE_NEGS = {"text": "baseline-bm25", "multimodal": "baseline"}


def discover_runs(models_root="models", note="paper"):
    """Paper analysis never discovers legacy training variants or archived checkpoints."""
    frame = (discover_profile_runs() if models_root == MODELS_ROOT
             else discover_all_runs(models_root, note=note))
    bases = active_dataset_bases()
    active = pd.Series([
        row.dataset_tag == bases[row.modality] + "_rephrased-in-context"
        or row.dataset_tag.startswith(bases[row.modality] + "_rephrased-in-context_")
        for row in frame.itertuples()
    ], index=frame.index, dtype=bool)
    if "reference" in frame:
        active |= frame["reference"]
    return frame[active & ~frame["style"].isin(RETIRED_INFONCE_STYLES)
                 & (frame["query_kind"] == ANALYSIS_QUERY_KIND)
                 & (frame["rephrase"] == ANALYSIS_REPHRASE)].copy()

TRAINING_PROFILE = training_profile()
if not TRAINING_PROFILE:
    raise RuntimeError("No resolved paper run. Complete bash paper.sh before running this notebook.")
NOTE = TRAINING_PROFILE["note"]
MODELS_ROOT = TRAINING_PROFILE["models_root"]
os.environ.pop("PAPER_ANALYSIS_EXTRA_PROFILES", None)
ANALYSIS_PROFILES = analysis_profiles()
print("Analysis model roots:", [p["models_root"] for p in ANALYSIS_PROFILES])
K_MAIN = 20                      # the paper's headline cutoff (full table, sorting)
# Text saturates -- at recall@20 most conditions sit above 0.9, and by 2026-09-03 the top
# ten validation cells were within 0.014 of each other at recall@10 as well -- so the text
# panels and the text hparam selection use recall@5 (top-ten spread 0.019). Image is far
# from saturation and stays at @20.
K_TEXT = 5
K_IMAGE = 20
KS = (1, 5, 10, 20, 100)


def k_for(modality):
    return K_TEXT if modality == "text" else K_IMAGE
FIG_DIR = os.environ.get("PAPER_FIG_DIR", "paper/figs")
os.makedirs(FIG_DIR, exist_ok=True)
WORK_DIR = os.environ["PAPER_ANALYSIS_DIR"] if "PAPER_ANALYSIS_DIR" in os.environ else "analysis/figs"
os.makedirs(WORK_DIR, exist_ok=True)

## Refresh human-matched data and retrieval evaluations

Downloads both original and both backfill spreadsheets on every Run All. The count cap
is the largest bin among counts 6+, separately by modality, with seed 42. Fixed
rephrasing examples are preserved and excluded by positive product. Changed inputs
trigger fresh inference on existing checkpoints; unchanged predictions are reused.


In [ ]:
from pathlib import Path
import subprocess

human_refresh_root = Path.cwd()
subprocess.run(
    [str(human_refresh_root / '.venv/bin/python'), '-B', '-u', 'analysis/refresh_human.py'],
    cwd=human_refresh_root, check=True,
)
with open('analysis/human_refresh_status.json') as handle:
    human_refresh_status = json.load(handle)
print('Human-matched refresh:', human_refresh_status['predictions'])
display(pd.DataFrame(human_refresh_status['summary']).T)


## 1. Conditions and health

Columns: modality, style, query_kind, V, easy. `V`/`easy` are None/20 for styles that never
see the measured distance. The merge is on all five keys, so the easy-ablation variants of
one (style, V) stay distinct.

In [ ]:
from utils.paper_analysis import parse_conditions, human_conditions as load_human_conditions


def analysis_conditions(paper_sh="paper.sh"):
    """Only rephrased training/evaluation conditions belong in this notebook."""
    frame = parse_conditions(paper_sh)
    return frame[(frame["query_kind"] == ANALYSIS_QUERY_KIND) & (frame["rephrase"] == ANALYSIS_REPHRASE)].copy()


def easy_of(extra):
    for token in (extra or "").split(","):
        if token.startswith("easy="):
            return int(token[len("easy="):])
    return 20


conditions = analysis_conditions()
conditions = conditions[~conditions["extra"].fillna("").str.contains("split=val")].copy()
conditions["easy"] = conditions["extra"].map(easy_of)
conditions = conditions.drop(columns="extra").drop_duplicates().reset_index(drop=True)
conditions["V"] = conditions["V"].astype("float64")

runs = discover_runs(MODELS_ROOT, note=NOTE)
runs = runs[runs["query_kind"] == ANALYSIS_QUERY_KIND].copy()
runs["easy"] = runs["easy"].fillna(20).astype(int)
matched = conditions.merge(runs, on=["modality", "style", "query_kind", "V", "easy",
                                    "negs", "mining", "seed", "rephrase", "order"], how="left", validate="one_to_one")
for column in ("has_preds", "has_human_preds"):
    matched[column] = matched[column].fillna(False).astype(bool)

health = health_check(matched)
print(f"{len(conditions)} conditions | {int(health['healthy'].sum())} usable | "
      f"{int((~health['healthy']).sum())} not")
display(health[~health["healthy"]][["modality", "style", "query_kind", "V", "n_queries", "problem"]])
if not health["healthy"].all():
    raise RuntimeError("Synthetic predictions are incomplete or stale; resume bash paper.sh")
usable = matched[health["healthy"].to_numpy()].reset_index(drop=True)

# Human predictions were refreshed from the current matched datasets above.
human_conditions = load_human_conditions()
human_conditions["V"] = pd.to_numeric(human_conditions["V"], errors="coerce")
human_conditions["easy"] = human_conditions["extra"].map(easy_of)
human_conditions = human_conditions.drop(columns="extra").drop_duplicates()
human_matched = human_conditions.merge(runs, on=["modality", "style", "query_kind", "V", "easy",
                                              "negs", "mining", "seed", "rephrase", "order"], how="left", validate="one_to_one")
for column in ("has_preds", "has_human_preds"):
    human_matched[column] = human_matched[column].fillna(False).astype(bool)
human_health = health_check(human_matched, min_queries=1, preds_subdir="preds_human")
print(f"{int(human_health['healthy'].sum())} conditions with usable human-query preds | "
      f"{int((~human_health['healthy']).sum())} without")
usable_human = human_matched[human_health["healthy"].to_numpy()].reset_index(drop=True)
if not human_health["healthy"].all():
    raise RuntimeError("Human predictions are incomplete or stale; rerun the refresh cell")


In [ ]:
# In-context rephrasings (paper.sh rephrase=in-context, 2026-09-11) are the same rephrased
# main-grid rows retrained on the _rephrased-in-context dataset and scored on its test split
# and on the _human-in-context set. They show as two more query kinds, suffixed "-ic", beside
# the plain rephrased and human bars; train_kind stays "rephrased" so they share its hparams.
IC_SUFFIX = "-ic"


def scored_kind(row, preds_subdir):
    base = "human" if preds_subdir == "preds_human" else row.query_kind
    return base + IC_SUFFIX if row.rephrase == "in-context" else base


def condition_metrics(row, preds_subdir="preds"):
    """One results row. train_kind is the query kind the model trained on and query_kind the
    kind it is scored on; they differ for human queries, which every model is scored on, and
    carry the -ic suffix for in-context models."""
    meta = json.load(open(os.path.join(row.run_dir, preds_subdir, "meta.json")))
    out = {"modality": row.modality, "style": row.style,
           "query_kind": scored_kind(row, preds_subdir),
           "train_kind": row.query_kind, "rephrase": row.rephrase,
           "V": row.V, "easy": row.easy, "negs": row.negs, "mining": row.mining,
           "order": row.order, "seed": row.seed, "run_dir": row.run_dir,
           "n_queries": meta["n_queries"], "reference": row.reference,
           "evaluation_dataset": meta["args"]["dataset"],
           "evaluation_corpus": (meta["args"]["distractor_dataset"] if preds_subdir == "preds_human"
                                 else meta["args"]["dataset"])}
    if preds_subdir == "preds_human":
        out["human_refresh_signature"] = meta["human_refresh_signature"]
    for k in KS:
        out[f"recall@{k}"] = meta["metrics"][f"recall@{k}"]
    return out

results = pd.DataFrame([condition_metrics(r) for r in usable.itertuples()]
                       + [condition_metrics(r, "preds_human") for r in usable_human.itertuples()])
results["query_kind"] = pd.Categorical(results["query_kind"],
                                       ["rephrased", "human",
                                        "rephrased" + IC_SUFFIX, "human" + IC_SUFFIX], ordered=True)
# One column that already holds each row's modality-appropriate cutoff, so the ablation
# pivots below stay a single metric column instead of one per modality.
results["recall@k_mod"] = [row[f"recall@{k_for(row['modality'])}"]
                           for _, row in results.iterrows()]
print(f"{len(results)} conditions loaded")
source_summary = (results.groupby(["modality", "query_kind", "reference"], observed=True)
                  .agg(models=("run_dir", "nunique"), queries=("n_queries", "first")))
display(source_summary)

# Check the full evaluation population before comparing training strategies.
import hashlib

population_signatures = {}
for row in results.itertuples():
    subdir = "preds_human" if row.query_kind.startswith("human") else "preds"
    directory = Path(row.run_dir) / subdir
    with (directory / "queries.jsonl").open() as handle:
        queries = [json.loads(line) for line in handle if line.strip()]
    triplets = []
    if subdir == "preds":
        with (directory / "triplets.jsonl").open() as handle:
            triplets = [json.loads(line) for line in handle if line.strip()]
    signature = (
        hashlib.sha256((directory / "corpus.jsonl").read_bytes()).hexdigest(),
        tuple((q["query_id"], q["query"], tuple(q["positive_corpus_ids"])) for q in queries),
        tuple((t["query_id"], t["positive_corpus_id"], t["negative_corpus_id"]) for t in triplets),
    )
    key = (row.modality, row.query_kind)
    if key in population_signatures and signature != population_signatures[key]:
        raise ValueError(f"Different evaluation population: {row.run_dir}/{subdir}")
    population_signatures[key] = signature
print("Verified common queries, positives, corpus, and synthetic hard-negative pairs for every main model.")
results[results["query_kind"] == "rephrased-ic"].to_csv(
    os.path.join(WORK_DIR, "main_results_models.tsv"), sep="\t", index=False)


## 2. Validation sweeps

Margin-MSE selects easy-negative distance and V jointly over a 3×3 grid, separately for each negative source. Graded InfoNCE and SigLIP use one-dimensional V sweeps. NV-Retriever selects its mining settings on validation recall. Text uses Recall@5 and images use Recall@20; test results do not enter selection.


In [ ]:
# The easy x V grid lives in preds_val/, so it is read from the run dirs rather than reused
# from `results` (which loads test preds). Discover rephrased validation runs directly.
EASY_LEVELS = [10, 20, 40]
V_LEVELS = [20.0, 40.0, 80.0]
INFONCE_GRADED_STYLE = "infonce-ours-v18"
GRADED_V_LEVELS = [10.0, 20.0, 40.0, 80.0]
# SigLIP retains its exponential soft target and one-dimensional validation sweep.
SIGLIP_GRADED_STYLE = "siglip-v3"


def val_condition(run):
    meta_path = os.path.join(run.run_dir, "preds_val", "meta.json")
    if not os.path.exists(meta_path):
        return None
    meta = json.load(open(meta_path))
    if (run.rephrase != ANALYSIS_REPHRASE or meta["split"] != "validation"
            or "_rephrased-in-context" not in meta["args"]["dataset"]):
        raise ValueError(f"Ineligible validation predictions: {run.run_dir}")
    return {"modality": run.modality, "style": run.style, "query_kind": run.query_kind,
            "V": run.V, "easy": run.easy, "negs": run.negs, "mining": run.mining,
            "rephrase": run.rephrase, "order": run.order, "seed": run.seed,
            "recall@k_mod": meta["metrics"][f"recall@{k_for(run.modality)}"]}


val_runs = discover_runs(MODELS_ROOT, note=NOTE)
val_runs = val_runs[val_runs["query_kind"] == ANALYSIS_QUERY_KIND].copy()
val_runs["easy"] = val_runs["easy"].fillna(20).astype(int)
validation_conditions = analysis_conditions()
validation_conditions = validation_conditions[validation_conditions["extra"].str.contains("split=val")].copy()
validation_conditions["easy"] = validation_conditions["extra"].map(easy_of)
validation_keys = ["modality", "style", "query_kind", "V", "easy", "negs", "mining", "seed", "rephrase", "order"]
val_runs = validation_conditions[validation_keys].drop_duplicates().merge(
    val_runs, on=validation_keys, how="left", validate="one_to_one")
if val_runs.run_dir.isna().any():
    raise RuntimeError("A declared validation checkpoint is missing")
ablation = pd.DataFrame([row for row in (val_condition(r) for r in val_runs.itertuples())
                         if row is not None and row["seed"] == 42])
# Sweep panels use the same query kind as the rest of the notebook.
ABLATION_KIND = ANALYSIS_QUERY_KIND
ablation = ablation[ablation["query_kind"] == ABLATION_KIND]
print(f"{len(ablation)} validation-split {ABLATION_KIND} conditions loaded")


In [ ]:
grid = ablation[ablation["style"].isin(["ours-mse-batched", "mse-mined"])
                & (ablation["order"] == "") & ablation["easy"].isin(EASY_LEVELS) & ablation["V"].isin(V_LEVELS)]
# Margin-MSE uses clip(easy / V, 0, 1); each negative source gets its own winner.
MSE_SWEEPS = {"GOLD graded": ("ours-mse-batched", "labeled"),
              "Margin-MSE / Baseline": ("mse-mined", "baseline"),
              "Margin-MSE / GOLD": ("mse-mined", "labeled"),
              "Margin-MSE / NV": ("mse-mined", "mined")}
for modality in ["text", "multimodal"]:
    sub = grid[grid["modality"] == modality]
    tables = {label: sub[(sub["style"] == style) & (sub["negs"] == (BASELINE_NEGS[modality] if negs == "baseline" else negs))]
              .pivot_table(index="easy", columns="V", values="recall@k_mod")
              .reindex(index=EASY_LEVELS, columns=V_LEVELS)
              for label, (style, negs) in MSE_SWEEPS.items()}
    full = [s for s, t in tables.items() if t.notna().to_numpy().all()]
    sparse = [s for s in tables if s not in full]

    if not full:
        print(f"{modality}: in-context easy × V sweep pending")
        continue

    # One colour scale across the row, so cells are comparable between styles and not just
    # within one heatmap.
    lo = min(tables[s].to_numpy().min() for s in full)
    hi = max(tables[s].to_numpy().max() for s in full)
    fig, axes = plt.subplots(1, len(full), figsize=(4.0 * len(full), 3.8), squeeze=False)
    for ax, style in zip(axes[0], full):
        table = tables[style]
        values = table.to_numpy()
        im = ax.imshow(values, cmap="viridis", vmin=lo, vmax=hi, aspect="auto")
        ax.grid(False)  # the seaborn whitegrid theme would draw rules across the cells
        best = values.argmax()
        for i, easy in enumerate(EASY_LEVELS):
            for j, v in enumerate(V_LEVELS):
                value = table.loc[easy, v]
                ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=9,
                        fontweight="bold" if i * len(V_LEVELS) + j == best else "normal",
                        color="white" if value < (lo + hi) / 2 else "black")
        ax.set_xticks(range(len(V_LEVELS)), [str(int(v)) for v in V_LEVELS])
        ax.set_yticks(range(len(EASY_LEVELS)), [str(e) for e in EASY_LEVELS])
        ax.set_xlabel("V")
        ax.set_ylabel("easy-negative distance")
        ax.set_title(style, fontsize=10)
    fig.colorbar(im, ax=axes[0], fraction=0.025, label=f"Recall@{k_for(modality)} (val)")
    # The selection metric differs by modality -- Recall@5 for text, Recall@20 for image,
    # via k_for -- so name it in the title instead of leaving it to the colourbar alone.
    fig.suptitle(f"{'text' if modality == 'text' else 'image'} \u2014 easy x V "
                 f"({ABLATION_KIND}-in-context, validation split)\n"
                 f"hparams selected on Recall@{k_for(modality)}; bold cell = argmax",
                 fontweight="bold", y=1.10)
    fig.savefig(os.path.join(WORK_DIR, f"easy_v_ablation_{modality}.png"), dpi=150,
                bbox_inches="tight")
    plt.show()
    if sparse:
        print(f"{modality}: incomplete grid, not drawn -- "
              + ", ".join(f"{s} ({int(tables[s].notna().to_numpy().sum())}/9)" for s in sparse))


In [ ]:
# infonce-ours-v18 has no easy axis: random rows keep ordinary InfoNCE,
# while measured negatives have distance-weighted inclusion. Its sweep is one-dimensional in V, so it is drawn as a line
# per modality with infonce-mined (the ungraded control, no V) on the same validation split.
# Retired comparison: ours-infonce-margin is no longer loaded or plotted.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = ablation[(ablation["modality"] == modality) & (ablation["negs"] == "labeled")
                   & (ablation["rephrase"] == ANALYSIS_REPHRASE)]
    sweep = (sub[sub["style"] == INFONCE_GRADED_STYLE].set_index("V")["recall@k_mod"]
             .reindex(GRADED_V_LEVELS))
    ax.plot(GRADED_V_LEVELS, sweep.to_numpy(), marker="o", label=INFONCE_GRADED_STYLE)
    if sweep.notna().any():
        best_v = sweep.idxmax()
        ax.annotate(f"V={int(best_v)}: {sweep[best_v]:.3f}", (best_v, sweep[best_v]),
                    textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    ref = sub[(sub["style"] == "infonce-mined") & sub["V"].isna()]
    if not ref.empty:
        ax.axhline(ref["recall@k_mod"].iloc[0], linestyle="--", linewidth=1,
                   label="infonce-mined")
    ax.set_xscale("log", base=2)
    ax.set_xticks(GRADED_V_LEVELS, [str(int(v)) for v in GRADED_V_LEVELS])
    ax.set_xlabel("V")
    ax.set_ylabel(f"Recall@{k_for(modality)} (val)")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
    ax.legend(fontsize=8)
    missing = [int(v) for v in GRADED_V_LEVELS if pd.isna(sweep.get(v))]
    if missing:
        ax.text(0.02, 0.02, f"missing V: {missing}", transform=ax.transAxes,
                fontsize=8, color="0.3")
fig.suptitle(f"{INFONCE_GRADED_STYLE} \u2014 V sweep ({ABLATION_KIND}, validation split); "
             f"V selected on Recall@{K_TEXT} text / Recall@{K_IMAGE} image",
             fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(WORK_DIR, "v_sweep_infonce_ours_v18.png"), dpi=150, bbox_inches="tight")
plt.show()

# siglip-v3: the same one-dimensional V sweep, on the in-context validation split (the paper's
# rephrased kind), with siglip-mined scored on that split as the ungraded reference.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = ablation[(ablation["modality"] == modality) & (ablation["negs"] == "labeled")
                   & (ablation["rephrase"] == "in-context") & (ablation["order"] == "")]
    sweep = (sub[sub["style"] == SIGLIP_GRADED_STYLE].set_index("V")["recall@k_mod"]
             .reindex(GRADED_V_LEVELS))
    # explicit limits: a log axis cannot autoscale on an all-NaN sweep
    ax.set_xlim(GRADED_V_LEVELS[0] / 1.5, GRADED_V_LEVELS[-1] * 1.5)
    ax.plot(GRADED_V_LEVELS, sweep.to_numpy(), marker="o", label=SIGLIP_GRADED_STYLE)
    if sweep.notna().any():
        best_v = sweep.idxmax()
        ax.annotate(f"V={int(best_v)}: {sweep[best_v]:.3f}", (best_v, sweep[best_v]),
                    textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    ref = sub[(sub["style"] == "siglip-mined") & sub["V"].isna()]
    if not ref.empty:
        ax.axhline(ref["recall@k_mod"].mean(), linestyle="--", linewidth=1, label="siglip-mined")
    ax.set_xscale("log", base=2)
    ax.set_xticks(GRADED_V_LEVELS, [str(int(v)) for v in GRADED_V_LEVELS])
    ax.set_xlabel("V")
    ax.set_ylabel(f"Recall@{k_for(modality)} (val)")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
    ax.legend(fontsize=8)
    missing = [int(v) for v in GRADED_V_LEVELS if pd.isna(sweep.get(v))]
    if missing:
        ax.text(0.02, 0.02, f"missing V: {missing}", transform=ax.transAxes,
                fontsize=8, color="0.3")
fig.suptitle(f"{SIGLIP_GRADED_STYLE} \u2014 V sweep (in-context {ABLATION_KIND}, validation split); "
             f"V selected on Recall@{K_TEXT} text / Recall@{K_IMAGE} image",
             fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(WORK_DIR, "v_sweep_siglip_v3.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# NV-Retriever mining sweep for the infonce-mined baseline (rephrased, validation split):
# candidate depth x TopK-PercPos percentile. The loss has no hparams, so the sweep is over how
# the negative is mined; every cell is its own mine_hard_negs.py --variant dataset,
# k<top-k>_p<percentile>, or k<top-k>_none for the unfiltered top-k. The swept cells are
# utils.training_plan.NV_VARIANTS, the same list the runner selected from; grid positions
# outside that list were never mined. A swept cell without preds_val is drawn as a hatched
# placeholder.
from utils.training_plan import NV_VARIANTS

NV_TOP_KS = [10, 100]
NV_PERCENTILES = [None, 95, 90, 80, 70]   # None = no filter
NV_REPHRASES = [ANALYSIS_REPHRASE]


def nv_cell(variant):
    """(top-k, percentile) of a mining variant tag; percentile None is the unfiltered top-k."""
    top_k, _, pct = variant.partition("_")
    return int(top_k[1:]), (None if pct == "none" else int(pct[1:]))


def nv_variant(top_k, pct, rephrase):
    """The mine_hard_negs.py --variant tag of a cell."""
    return f"k{top_k}_" + ("none" if pct is None else f"p{pct}")


NV_CELLS = [nv_cell(v) for v in NV_VARIANTS]

# Seed 42 only: the selected cell also has seed-43/44 val rows (baseline table).
nv_runs = val_runs[(val_runs["style"] == "infonce-mined") & (val_runs["negs"] == "mined")
                   & (val_runs["query_kind"] == ABLATION_KIND) & (val_runs["order"] == "")
                   & (val_runs["seed"] == 42) & val_runs["mining"].isin(NV_VARIANTS)]
nv_values = {}
for run in nv_runs.itertuples():
    row = val_condition(run)
    if row is not None:
        nv_values[run.modality, run.rephrase, nv_cell(run.mining)] = row["recall@k_mod"]
print(f"{len(nv_values)} of {2 * len(NV_REPHRASES) * len(NV_VARIANTS)} mining-sweep cells have preds_val")


def nv_val_argmax(modality, rephrase):
    """(top-k, percentile) with the best validation recall; raises on an incomplete grid."""
    cells = {c: nv_values[modality, rephrase, c] for c in NV_CELLS if (modality, rephrase, c) in nv_values}
    absent = [nv_variant(*c, rephrase) for c in NV_CELLS if c not in cells]
    if absent:
        raise ValueError(f"{modality} {rephrase or 'plain'}: mining sweep incomplete, no preds_val for {absent}")
    return max(cells, key=cells.__getitem__)


fig, axes = plt.subplots(len(NV_REPHRASES), 2, figsize=(10, 3.2 * len(NV_REPHRASES)), squeeze=False)
for (ax, modality), rephrase in [((ax, m), r) for r, row in zip(NV_REPHRASES, axes) for ax, m in zip(row, ["text", "multimodal"])]:
    values = np.full((len(NV_TOP_KS), len(NV_PERCENTILES)), np.nan)
    for i, top_k in enumerate(NV_TOP_KS):
        for j, pct in enumerate(NV_PERCENTILES):
            if (modality, rephrase, (top_k, pct)) in nv_values:
                values[i, j] = nv_values[modality, rephrase, (top_k, pct)]
    done = ~np.isnan(values)
    lo, hi = (values[done].min(), values[done].max()) if done.any() else (0.0, 1.0)
    ax.imshow(np.where(done, values, lo), cmap="viridis", vmin=lo, vmax=hi, aspect="auto")
    ax.grid(False)
    best = np.nanargmax(values) if done.any() else None
    for i, top_k in enumerate(NV_TOP_KS):
        for j, pct in enumerate(NV_PERCENTILES):
            if done[i, j]:
                ax.text(j, i, f"{values[i, j]:.3f}", ha="center", va="center", fontsize=9,
                        fontweight="bold" if i * len(NV_PERCENTILES) + j == best else "normal",
                        color="white" if values[i, j] < (lo + hi) / 2 else "black")
            else:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="0.9",
                                           edgecolor="0.6", hatch="///", linewidth=0))
                ax.text(j, i, "pending" if (top_k, pct) in NV_CELLS else "not swept",
                        ha="center", va="center", fontsize=7, color="0.3")
    ax.set_xticks(range(len(NV_PERCENTILES)), ["no filter" if p is None else f"p{p}" for p in NV_PERCENTILES])
    ax.set_yticks(range(len(NV_TOP_KS)), [f"top-{k}" for k in NV_TOP_KS])
    ax.set_xlabel("TopK-PercPos percentile")
    ax.set_title(("text" if modality == "text" else "image") + (f", {ABLATION_KIND}{IC_SUFFIX}" if rephrase else f", {ABLATION_KIND}"), fontsize=10)
fig.suptitle(f"infonce-mined on retrieval-mined negatives — mining sweep "
             f"(validation split; Recall@{K_TEXT} text / Recall@{K_IMAGE} image)",
             fontweight="bold", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(WORK_DIR, "nv_mining_sweep.png"), dpi=150, bbox_inches="tight")
plt.show()


## 3. Baseline, NV-Retriever, GOLD-mined, and GOLD-reweighted

All four strategies share held-out queries and corpus within each modality.

Bars show mean recall across three seeds. Arrows compare baseline → GOLD-mined → GOLD-reweighted.


In [ ]:
from utils.paper_analysis import parse_conditions

# Protocol pairs, in priority order: (ungraded mined loss, its graded counterpart).
PROTOCOL = {
    "infonce": ("infonce-mined", "infonce-ours-v18"),
    "mse":     ("mse-mined",     "ours-mse-batched"),
    "cosent":  ("cosent",        "ours-cosent"),
    # The exponential-target siglip-v3 replaces the retired ours-siglip variant.
    "siglip":  ("siglip-mined",  SIGLIP_GRADED_STYLE),
}
# The retrieval-mined (NV-Retriever) baseline: infonce-mined trained on the mine_hard_negs.py
# negatives instead of ours. One model per slot, drawn beside every family as the reference.
NV_MINED = "nv-mined"
ROLE = {NV_MINED: "NV-Retriever"}
# Mining config the nv-mined baseline uses per slot, selected on the sweep in section 2 the
# way SELECTED is for the graded styles: the entry must equal the sweep argmax and the
# paper.sh row must carry the same mining= tag. Keyed by (modality, train kind, rephrase
# variant): each rephrasing uses its own validation sweep.
NV_SELECTED = {
    (modality, ANALYSIS_QUERY_KIND, ANALYSIS_REPHRASE):
        (nv_variant(*nv_val_argmax(modality, ANALYSIS_REPHRASE), ANALYSIS_REPHRASE)
         if (nv_runs["modality"] == modality).any() else None)
    for modality in ("text", "multimodal")
}
BASELINE = "baseline"
ROLE[BASELINE] = "baseline"


def baseline_groups(modality):
    return [BASELINE]

for ungraded, graded in PROTOCOL.values():
    ROLE[ungraded], ROLE[graded] = "ungraded", "graded"
main_grid = conditions[conditions["negs"] == "labeled"].copy()
QK_ORDER = [ANALYSIS_QUERY_KIND + IC_SUFFIX, "human" + IC_SUFFIX]
RANK_KINDS = [ANALYSIS_QUERY_KIND + IC_SUFFIX]
qk_colour = dict(zip(QK_ORDER, sns.color_palette("colorblind", len(QK_ORDER))))


def train_kind_of(kind):
    return ANALYSIS_QUERY_KIND if kind.startswith("human") else kind.removesuffix(IC_SUFFIX)


def rephrase_of(kind):
    return "in-context" if kind.endswith(IC_SUFFIX) else ""


# Read the resolved validation choices used for main training.
selected_rows = main_grid[["modality", "style", "query_kind", "V", "easy"]].drop_duplicates()
if selected_rows.duplicated(["modality", "style", "query_kind"]).any():
    raise ValueError("paper.sh declares multiple hyperparameter choices for a main result")
SELECTED = {(r.modality, r.style, r.query_kind): {"V": r.V, "easy": r.easy}
            for r in selected_rows.itertuples()}
HUMAN_MAIN_GRID = human_conditions[human_conditions["negs"] == "labeled"].copy()
HUMAN_SELECTED = {(r.modality, r.style, r.query_kind): {"V": r.V, "easy": r.easy}
                  for r in HUMAN_MAIN_GRID.itertuples()}

# The sweep grid each style is selected over. infonce-ours-v18 has no easy axis, so its
# grid is V alone at the default easy (see section 2).
SWEEP_GRID = {
    "mse-mined":        [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    "ours-mse-batched": [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    "ours-siglip":      [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    INFONCE_GRADED_STYLE:           [(20, V) for V in GRADED_V_LEVELS],
    SIGLIP_GRADED_STYLE:    [(20, V) for V in GRADED_V_LEVELS],
}
# The rephrase variant each style's sweep ran on; the in-context rows of every other style
# reuse the plain-rephrased selection, siglip-v3 was only ever swept in-context.
SWEEP_REPHRASE = {style: ANALYSIS_REPHRASE for style in SWEEP_GRID}


def val_argmax(modality, style, kind):
    """(easy, V) with the best validation recall over the style's sweep grid for this query
    kind; raises if any grid cell has no preds_val, since a selection needs the whole grid."""
    cells = {}
    for run in val_runs[(val_runs["modality"] == modality) & (val_runs["style"] == style)
                        & (val_runs["query_kind"] == kind) & (val_runs["negs"] == "labeled")
                        & (val_runs["rephrase"] == ANALYSIS_REPHRASE) & (val_runs["seed"] == 42)].itertuples():
        row = val_condition(run)
        if row is not None:
            cells[int(run.easy), float(run.V)] = row["recall@k_mod"]
    grid = SWEEP_GRID[style]
    absent = [cell for cell in grid if cell not in cells]
    if absent:
        raise ValueError(f"{modality} {style} {kind}: sweep incomplete, no preds_val for {absent}")
    return max(grid, key=lambda cell: cells[cell])


selection_changes = []
for (modality, style, kind), chosen in SELECTED.items():
    if style in SWEEP_GRID:
        best_easy, best_v = val_argmax(modality, style, kind)
        if (chosen["easy"], chosen["V"]) != (best_easy, best_v):
            selection_changes.append({"modality": modality, "style": style,
                                      "paper_easy": chosen["easy"], "paper_V": chosen["V"],
                                      "validation_easy": best_easy, "validation_V": best_v})
if selection_changes:
    raise ValueError(f"Resolved training choices differ from validation winners: {selection_changes}")


def main_row(modality, style, kind):
    """The test-split results row for one protocol slot, or (None, why it is missing)."""
    train_kind = train_kind_of(kind)
    source_grid = HUMAN_MAIN_GRID if kind.startswith("human") else main_grid
    cond = source_grid[(source_grid["modality"] == modality) & (source_grid["style"] == style)
                       & (source_grid["query_kind"] == train_kind) & (source_grid["rephrase"] == rephrase_of(kind))]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    if style in SWEEP_GRID:
        chosen = (HUMAN_SELECTED if kind.startswith("human") else SELECTED)[modality, style, train_kind]
        if chosen is None:
            return None, "hparam selection pending"
    if style in SWEEP_GRID and (c["easy"], float(c["V"])) != (chosen["easy"], float(chosen["V"])):
        raise ValueError(f"{modality} {style} {train_kind}: paper.sh row has easy={c['easy']}, "
                         f"V={int(c['V'])}, SELECTED says {chosen}")
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == "labeled") & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


def trials_or_why(hit, cond):
    """(trials, note) for a condition: its healthy trial rows, or (None, why) when there are
    none. When fewer seeds than paper.sh declares are done, the rows are returned and the
    note says so; the bar is drawn from the seeds that exist and carries the note."""
    declared = int(cond["seed"].nunique())
    if hit.empty:
        return None, "no preds"
    if len(hit) < declared:
        return hit, f"{len(hit)}/{declared} trials"
    return hit, None


def trial_stats(frame, k):
    """mean over a condition's trials, and the number of trials."""
    values = frame[f"recall@{k}"].to_numpy(dtype=float)
    return float(values.mean()), len(values)


def kinds_for(modality):
    """Rephrased queries and the human study for the in-context models."""
    grid = main_grid[main_grid["modality"] == modality]
    return list(QK_ORDER) if (grid["rephrase"] == "in-context").any() else []


rows, missing, incomplete = [], {}, {}
for modality in ["text", "multimodal"]:
    for styles in PROTOCOL.values():
        for style in styles:
            for kind in kinds_for(modality):
                trials, note = main_row(modality, style, kind)
                if trials is None:
                    missing[modality, style, kind] = note
                    continue
                rows.append(trials)
                if note:
                    incomplete[modality, style, kind] = note
main = pd.concat(rows).reset_index(drop=True)
n_slots = len(main.drop_duplicates(["modality", "style", "query_kind"]))
print(f"{n_slots} protocol slots with results ({len(main)} trials), {len(missing)} missing, "
      f"{len(incomplete)} with fewer seeds than declared")


def ordered_groups(modality, styles, value_of):
    """Groups sorted by mean value over RANK_KINDS, best first; missing last."""
    kinds = kinds_for(modality)
    order_kinds = [k for k in RANK_KINDS if k in kinds]

    def key(style):
        vals = [value_of(modality, style, k) for k in order_kinds]
        return -sum(vals) / len(vals) if vals and all(v is not None for v in vals) else float("inf")
    return sorted(styles, key=key), kinds


def cell_x(i, j, width, kinds):
    """x of the bar for group i, kind j."""
    return i + (j - (len(kinds) - 1) / 2) * width


def label_cell(ax, x, text):
    """Vertical note on one bar cell (a missing reason, or a fewer-seeds-than-declared note)."""
    ax.text(x, 0.5, text, rotation=90, ha="center", va="center", fontsize=7,
            color="0.3", transform=ax.get_xaxis_transform(), zorder=3)


def draw_missing(ax, i, j, width, kinds, reason):
    """Hatched placeholder for one empty (group, kind) cell, labeled with its reason."""
    x = cell_x(i, j, width, kinds)
    ax.axvspan(x - width * 0.475, x + width * 0.475, facecolor="0.9", edgecolor="0.6",
               hatch="///", linewidth=0, zorder=0)
    label_cell(ax, x, reason)


grid_all_mined = analysis_conditions()
grid_all_mined = grid_all_mined[~grid_all_mined["extra"].fillna("").str.contains("split=val")
                                & (grid_all_mined["negs"] == "mined")]


def nv_row(modality, kind, style="infonce-mined"):
    """The nv-mined test row for one slot -- `style` (an ungraded loss) trained on the NV-mined
    negatives at NV_SELECTED's mining variant -- or (None, why it is missing)."""
    train_kind = train_kind_of(kind)
    rephrase = rephrase_of(kind)
    chosen = NV_SELECTED[modality, train_kind, rephrase]
    if chosen is None:
        return None, "missing NV selection"
    best = nv_variant(*nv_val_argmax(modality, rephrase), rephrase)
    if chosen != best:
        raise ValueError(f"{modality} {rephrase or 'plain'}: NV_SELECTED says {chosen!r}, sweep argmax is {best!r}")
    cond = grid_all_mined[(grid_all_mined["modality"] == modality) & (grid_all_mined["query_kind"] == train_kind)
                          & (grid_all_mined["style"] == style) & (grid_all_mined["rephrase"] == rephrase_of(kind))]
    if cond.empty or cond["mining"].iloc[0] != chosen:
        return None, f"paper.sh row is not mining={chosen}"
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["negs"] == "mined") & (results["mining"] == chosen)
                  & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


grid_baseline = analysis_conditions()
grid_baseline = grid_baseline[~grid_baseline["extra"].fillna("").str.contains("split=val")
                          & grid_baseline["negs"].isin(["baseline", "baseline-bm25"])].copy()
grid_baseline["easy"] = grid_baseline["extra"].map(easy_of)


def baseline_row(modality, kind, style, negs=BASELINE):
    """The baseline-control test row for one slot -- `style` (an ungraded loss) on the baseline
    negatives -- or (None, why it is missing)."""
    if rephrase_of(kind) != ANALYSIS_REPHRASE:
        return None, "in-context only"
    negs = BASELINE_NEGS[modality] if negs == BASELINE else negs
    train_kind = train_kind_of(kind)
    cond = grid_baseline[(grid_baseline["modality"] == modality) & (grid_baseline["query_kind"] == train_kind)
                       & (grid_baseline["style"] == style) & (grid_baseline["negs"] == negs)
                       & (grid_baseline["rephrase"] == rephrase_of(kind))]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == negs) & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


def stats_of(modality, style, kind):
    """(mean, n) over a slot's trials, or None if the slot is missing."""
    trials = main[(main["modality"] == modality) & (main["style"] == style) & (main["query_kind"] == kind)]
    trials = None if trials.empty else trials
    return None if trials is None else trial_stats(trials, k_for(modality))


def recall_of(modality, style, kind):
    st = stats_of(modality, style, kind)
    return None if st is None else st[0]


def print_ranking(modality, groups, value_of, stats_for=None):
    """Groups best first by mean recall over RANK_KINDS, as three tables with the same rows
    and columns: the mean of every kind of the modality ('-' when that kind is missing),
    then the number of trials. A group missing a
    RANK_KINDS value is listed as missing in the first table and omitted from the others."""
    show_kinds = kinds_for(modality)
    order_kinds = [k for k in RANK_KINDS if k in show_kinds]
    w = 13

    def table(title, cell):
        print(f"  {modality}: {title}")
        print(f"    {'group':<26}" + "".join(f"{kind:>{w}}" for kind in show_kinds))
        for g in groups:
            if any(value_of(modality, g, kind) is None for kind in order_kinds):
                if title.startswith("Recall"):
                    print(f"    {g:<26}missing")
                continue
            print(f"    {g:<26}" + "".join(f"{cell(g, kind):>{w}}" for kind in show_kinds))

    def fmt(g, kind, pick):
        if value_of(modality, g, kind) is None:
            return "-"
        return pick(*stats_for(modality, g, kind))

    table(f"Recall@{k_for(modality)}", lambda g, kind: fmt(g, kind, lambda mean, n: f"{mean:.4f}"))
    if stats_for:
        table("n trials", lambda g, kind: fmt(g, kind, lambda mean, n: str(n)))


# Per family: the pair plus the family's baseline and nv-mined groups (its ungraded style on the
# baseline and on the NV-mined negatives). The shared labels resolve through the family's own
# styles, so missing notes are kept per family.
EXTRA_GROUPS = [BASELINE, NV_MINED]
extra_missing = {}
for family, pair in PROTOCOL.items():
    print(f"{family}:")
    ungraded = pair[0]

    def extra_row(modality, group, kind):
        if group == BASELINE:
            return baseline_row(modality, kind, ungraded, group)
        if group == NV_MINED:
            return nv_row(modality, kind, ungraded)
        raise ValueError(group)

    fam_missing, fam_incomplete = dict(missing), dict(incomplete)
    fam_stats_cache = {}
    for modality in ["text", "multimodal"]:
        for group in baseline_groups(modality) + [NV_MINED]:
            for kind in kinds_for(modality):
                trials, note = extra_row(modality, group, kind)
                if trials is None:
                    fam_missing[modality, group, kind] = note
                    extra_missing[family, modality, group, kind] = note
                    continue
                fam_stats_cache[modality, group, kind] = trial_stats(trials, k_for(modality))
                if note:
                    fam_incomplete[modality, group, kind] = note

    def fam_stats(modality, style, kind):
        if style in EXTRA_GROUPS:
            return fam_stats_cache[modality, style, kind] if (modality, style, kind) in fam_stats_cache else None
        return stats_of(modality, style, kind)

    def fam_recall(modality, style, kind):
        st = fam_stats(modality, style, kind)
        return None if st is None else st[0]

    fig, axes = plt.subplots(1, 2, figsize=(max(11, 3.2 * (len(pair) + len(EXTRA_GROUPS))), 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        groups, kinds = ordered_groups(modality, list(pair) + baseline_groups(modality) + [NV_MINED], fam_recall)
        print_ranking(modality, groups, fam_recall, fam_stats)
        width = 0.8 / len(kinds)
        for j, k in enumerate(kinds):
            xs = [cell_x(i, j, width, kinds) for i in range(len(groups))]
            sts = [fam_stats(modality, s, k) for s in groups]
            ys = [st[0] if st is not None else None for st in sts]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    draw_missing(ax, i, j, width, kinds, fam_missing[modality, s, k])
                elif (modality, s, k) in fam_incomplete:
                    label_cell(ax, xs[i], fam_incomplete[modality, s, k])
        if not any(fam_recall(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{'Baseline' if s == BASELINE else s}\n({ROLE[s]}, n={max(st[1] for st in (fam_stats(modality, s, k) for k in kinds) if st)})"
                            if any(fam_stats(modality, s, k) for k in kinds) else f"{s}\n({ROLE[s]})"
                            for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel(f"Recall@{k_for(modality)}")
    axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=5, frameon=False)
    fig.suptitle(f"{family}: baseline vs ungraded vs graded vs nv-mined (test split)", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(WORK_DIR, f"protocol_{family}.png"), dpi=150)
    plt.show()

if extra_missing:
    print("missing baseline / nv-mined slots:")
    for (family, modality, group, kind), why in sorted(extra_missing.items()):
        print(f"  {family:8} {modality:10} {group:14} {kind:12} {why}")
if missing:
    print("missing protocol slots:")
    for (modality, style, kind), why in sorted(missing.items()):
        print(f"  {modality:10} {style:20} {kind:10} {why}")


In [ ]:
from utils.human_freshness import prediction_problem
for human_result in results[results["query_kind"].astype(str).str.startswith("human")].itertuples():
    problem = prediction_problem(human_result.run_dir)
    current = json.load(open(os.path.join(human_result.run_dir, "preds_human/meta.json")))
    if problem or current["human_refresh_signature"] != human_result.human_refresh_signature:
        raise RuntimeError("Human results changed after loading; rerun the notebook from the refresh cell")

# Text recall against image recall, one point per (query kind, family, group) (2026-09-15).
# Colour is the family, marker the group, and the arrows are the changes the paper argues about:
# adding GOLD negatives (baseline -> ours-ungraded) and grading ours
# (ours-ungraded -> ours-graded). A family contributes a point only where both modalities have
# trials.
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import matplotlib.ticker as mticker

# Retain both evaluations for the appendix and human/synthetic comparisons.
# Figure 2a itself displays only the synthetic test results.
FIG2A_KINDS = [RANK_KINDS[0]]
PLOT_KINDS = [RANK_KINDS[0], "human" + IC_SUFFIX]
KIND_LINESTYLE = {PLOT_KINDS[0]: ":", PLOT_KINDS[1]: ":"}
KIND_LABEL = {PLOT_KINDS[0]: "Synthetic", PLOT_KINDS[1]: "Human (matched)"}
ROLE_MARKER = {"baseline": "o", "nv-mined": "^", "ours-ungraded": "s", "ours-graded": "D"}
# s is an area in points^2. Three groups sit at 70; the diamond is cut 30% from 70.
ROLE_SIZE = {"baseline": 70, "nv-mined": 70, "ours-ungraded": 70, "ours-graded": 49}
# The two groups that train on no labeled negative are drawn unfilled, in either query kind.
ROLE_FILLED = {"baseline": False, "nv-mined": False, "ours-ungraded": True, "ours-graded": True}
ROLE_EDGES = [("baseline", "ours-ungraded"),
              ("ours-ungraded", "ours-graded")]
# One label per family and per group, shared by this figure's legend and the appendix
# tables, so a figure and a table cannot call the same thing different names.
FAMILY_LABEL = {"infonce": "InfoNCE", "siglip": "SigLIP", "mse": "Margin-MSE",
                "cosent": "CoSENT"}
GROUP_LABEL = {"baseline": "Baseline", "nv-mined": "NV-Retriever",
               "ours-ungraded": "GOLD mined",
               "ours-graded": "GOLD reweighted (ours)"}
# Figure legends identify both GOLD variants as ours.
GROUP_LEGEND_LABEL = {**GROUP_LABEL, "ours-ungraded": "GOLD mined (ours)"}
PAIRED_GROUP_LABEL = GROUP_LEGEND_LABEL


def roles_for(modality):
    return list(ROLE_MARKER)


def role_edges_for(modality):
    return ROLE_EDGES
# The graded member plotted per family is the last of its PROTOCOL tuple (its newest member).
family_colour = dict(zip(PROTOCOL, sns.color_palette("colorblind", len(PROTOCOL))))
# Draw and list the families in this order; the colours stay keyed on PROTOCOL, so a family
# keeps the colour it has in every earlier render of this figure.
FAMILY_ORDER = ["infonce", "siglip", "mse", "cosent"]
assert set(FAMILY_ORDER) == set(PROTOCOL), sorted(set(PROTOCOL) ^ set(FAMILY_ORDER))


def role_trials(pair, role, modality, kind):
    """The section-3 trials behind one (kind, family, group) point, or None when the slot is empty."""
    ungraded, graded = pair[0], pair[-1]
    if role == BASELINE:
        return baseline_row(modality, kind, ungraded)[0]
    if role == "nv-mined":
        return nv_row(modality, kind, ungraded)[0]
    if role == "ours-ungraded":
        return main_row(modality, ungraded, kind)[0]
    if role == "ours-graded":
        return main_row(modality, graded, kind)[0]
    raise ValueError(role)


points, spans, absent = {}, {}, []
modality_values, modality_spans = {}, {}
for kind in PLOT_KINDS:
    for family in FAMILY_ORDER:
        pair = PROTOCOL[family]
        for role in ROLE_MARKER:
            recalls, ranges = {}, {}
            for modality in ("text", "multimodal"):
                trials = role_trials(pair, role, modality, kind)
                if trials is not None:
                    values = trials[f"recall@{k_for(modality)}"]
                    recalls[modality] = float(values.mean())
                    ranges[modality] = (float(values.min()), float(values.max()))
                    modality_values[kind, family, role, modality] = recalls[modality]
                    modality_spans[kind, family, role, modality] = ranges[modality]
            if len(recalls) == 2:
                points[kind, family, role] = (recalls["text"], recalls["multimodal"])
                spans[kind, family, role] = (ranges["text"], ranges["multimodal"])
            else:
                absent.append((kind, family, role, sorted(recalls)))

if absent:
    raise RuntimeError(f"Figure 2a has missing modality/group points: {absent}")
figure_rows = [
    {"query_kind": kind, "family": family, "group": role,
     "text_recall_at_5": x, "image_recall_at_20": y,
     "image_group": role, "text_reference": False}
    for (kind, family, role), (x, y) in points.items()
]
pd.DataFrame(figure_rows).to_csv(os.path.join(WORK_DIR, "text_vs_image_recall.csv"), index=False)

# Image recall on x, text recall on y.
plotted = {key: (image, text) for key, (text, image) in points.items()}
fig, ax = plt.subplots(figsize=(7.5, 4.131))
ax.set_axisbelow(True)
ax.grid(True, axis="both", linestyle=":", linewidth=0.6, alpha=0.7)
# Include every new Baseline result; the legacy random CoSENT control omitted positive pairs.
SCALE_EXCLUDE = set()
xs = [x for (kind, f, r), (x, _) in plotted.items()
      if kind in FIG2A_KINDS and (f, r) not in SCALE_EXCLUDE]
ys = [y for (kind, f, r), (_, y) in plotted.items()
      if kind in FIG2A_KINDS and (f, r) not in SCALE_EXCLUDE]
# Label evenly spaced subdivisions of the range the points actually occupy. The step
# comes from the range rather than a fixed list, so a panel whose points move or spread
# out keeps ticks across its whole width instead of over part of it.
def nice_ticks(lo, hi, target=8):
    """Round ticks at a 1/2/5 step covering [lo, hi], at most `target` of them. The step stays
    a multiple of 0.01 so every label is exact at the two decimals the formatter prints."""
    span = hi - lo
    if span <= 0:
        return [lo]
    step = max(10.0 ** np.floor(np.log10(span / target)), 0.01)
    step = next(step * m for m in (1, 2, 5, 10) if span / (step * m) <= target)
    return np.arange(np.ceil(lo / step) * step, hi + step / 2, step)


for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.set_xticks(nice_ticks(min(xs), max(xs)))
ax.set_yticks(nice_ticks(min(ys), max(ys)))
pad_x = max(0.03 * (max(xs) - min(xs)), 0.001)
pad_y = max(0.03 * (max(ys) - min(ys)), 0.001)
ax.set_xlim(min(xs) - pad_x, max(xs) + pad_x)
ax.set_ylim(min(ys) - pad_y, max(ys) + pad_y)
ax.set_autoscale_on(False)

for kind in FIG2A_KINDS:
    for family in FAMILY_ORDER:
        pair, colour = PROTOCOL[family], family_colour[family]
        for role, marker in ROLE_MARKER.items():
            if (kind, family, role) in plotted:
                x, y = plotted[kind, family, role]
                filled = ROLE_FILLED[role]
                ax.scatter([x], [y], marker=marker, s=ROLE_SIZE[role],
                           facecolor=colour if filled else "none",
                           edgecolor="white" if filled else colour,
                           linewidth=0.5 if filled else 1.2, zorder=3)

# Legend keys: loss-family colours in the first column, strategy markers in the second.
ax.legend(
    handles=[Line2D([], [], color=family_colour[f], linewidth=3, label=FAMILY_LABEL[f])
             for f in FAMILY_ORDER]
    + [Line2D([], [], color="0.35", marker=m, linewidth=0, markersize=np.sqrt(ROLE_SIZE[r]),
              markerfacecolor="0.35" if ROLE_FILLED[r] else "none", label=PAIRED_GROUP_LABEL[r].replace(" + ", "\n+ "))
       for r, m in reversed(list(ROLE_MARKER.items()))],
    loc="lower right", ncol=2, frameon=False, fontsize=8)
ax.set_xlabel(f"Image Recall@{K_IMAGE}")
ax.set_ylabel(f"BM25 text Recall@{K_TEXT}")
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"text_vs_image_recall.{ext}"), dpi=150)
plt.show()

# The numbers behind the figure, as the tabular main.tex inputs in the appendix. Only the
# tabular lives here; the table environment, caption and label are in main.tex. Rows follow the
# figure's family and group order, columns are (query kind x modality) at each modality's k.
# The best value in a column is bold, and the best within each family block is underlined, both
# computed here rather than typed into the tex.
TEX_FAMILY = FAMILY_LABEL
TEX_GROUP = GROUP_LABEL
TEX_KIND = KIND_LABEL
MAIN_TABLE_KIND = {PLOT_KINDS[0]: "Main", PLOT_KINDS[1]: "Human (Matched)"}
TEX_COLUMNS = [(kind, modality) for kind in PLOT_KINDS for modality in ("text", "multimodal")]


def column_value(family, role, column):
    """One cell's recall, or None when that (kind, family, group) slot has no point."""
    kind, modality = column
    key = kind, family, role, modality
    return modality_values[key] if key in modality_values else None


# Winners per column, overall and within each family block.
best_overall, best_in_family = {}, {}
for column in TEX_COLUMNS:
    scored = [(f, r) for f in FAMILY_ORDER for r in ROLE_MARKER
              if column_value(f, r, column) is not None]
    if scored:
        best_overall[column] = max(scored, key=lambda fr: column_value(*fr, column))
    for family in FAMILY_ORDER:
        rows = [r for r in ROLE_MARKER if column_value(family, r, column) is not None]
        if rows:
            best_in_family[family, column] = max(rows, key=lambda r: column_value(family, r, column))


def tex_cell(family, role, column):
    value = column_value(family, role, column)
    if value is None:
        return r"\textcolor{red}{pending}"
    text = f"{value:.3f}"
    if best_in_family.get((family, column)) == role:
        text = r"\underline{" + text + "}"
    if best_overall.get(column) == (family, role):
        text = r"\textbf{" + text + "}"
    # lowest and highest of the slot's seeds, in small type; leading zeros dropped so the
    # four numeric columns still fit the text block.
    kind, modality = column
    low, high = modality_spans[kind, family, role, modality]
    return text + r"\,{\tiny[" + f"{low:.3f}--{high:.3f}".replace("0.", ".") + "]}"


header = " & ".join(["Loss", "Group"] + [f"{MAIN_TABLE_KIND[k]} {'Text' if m == 'text' else 'Image'}"
                                         for k, m in TEX_COLUMNS])
lines = [r"\scriptsize", r"\setlength{\tabcolsep}{3pt}", r"\begin{tabular}{llrrrr}", r"    \toprule",
         f"    {header} \\\\",
         "    & & " + " & ".join(f"(Recall@{k_for(m)})" for _, m in TEX_COLUMNS) + r" \\",
         r"    \midrule"]
for family in FAMILY_ORDER:
    for position, role in enumerate(ROLE_MARKER):
        name = TEX_FAMILY[family] if position == 0 else ""
        lines.append("    " + " & ".join([name, TEX_GROUP[role]]
                                         + [tex_cell(family, role, c) for c in TEX_COLUMNS]) + r" \\")
    if family != FAMILY_ORDER[-1]:
        lines.append(r"    \midrule")
lines += [r"    \bottomrule", r"\end{tabular}"]
tex_path = os.path.join(FIG_DIR, "main_results_table.tex")
with open(tex_path, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"wrote {tex_path}")

print(f"{len(points)} of {len(PLOT_KINDS) * len(PROTOCOL) * len(ROLE_MARKER)} "
      f"(kind, family, group) points have both modalities")
for kind, family, role, have in absent:
    print(f"  {kind:14} {family:8} {role:14} only {have if have else 'neither modality'}")
print("outside the axis limits: " + ", ".join(f"{f}/{r}" for f, r in sorted(SCALE_EXCLUDE)))


### 3a. Human versus synthetic recall, by modality

Two plots use the **same core models as Figure 2a**, showing each of seeds 42, 43, and 44 separately: synthetic recall on x and matched-human recall on y. Each point pairs evaluations of the same checkpoint. Text uses Recall@5; images use Recall@20. Loss colours, strategy markers, upper-left legends, and linear tick conventions follow Figure 2a. All 60 text points and 48 image points, including baseline CoSENT, are included in the axis limits and in a single ordinary least-squares trendline with an intercept. The legend reports its descriptive R² across all points; seeds are not averaged. No training-strategy arrows are drawn.

Synthetic queries are rephrased-in-context; human queries use the current attribute-count-matched evaluation. Outputs: `analysis/figs/human_vs_synthetic_{text,image}_recall.{png,pdf,csv}`; fit statistics: `analysis/figs/human_vs_synthetic_recall_fits.csv`.

Compact paper panels and the Pearson r / Spearman ρ table are generated from the same points in `paper/figs/human_vs_synthetic_{text,image}_recall.pdf` and `paper/figs/human_synthetic_correlations.tex`.

In [ ]:
# Pair synthetic and matched-human recall from the same checkpoint and training seed.
# Model selection and styling come from Figure 2a; plot every seed without averaging.
hs_kinds = (RANK_KINDS[0], "human" + IC_SUFFIX)
hs_figures, hs_summaries, hs_fit_rows, hs_plot_limits = {}, {}, [], {}
for modality, name in (("text", "text"), ("multimodal", "image")):
    hs_rows = []
    for family in FAMILY_ORDER:
        for role in roles_for(modality):
            hs_trials = [role_trials(PROTOCOL[family], role, modality, kind) for kind in hs_kinds]
            if any(trials is None or trials.empty for trials in hs_trials):
                raise ValueError(f"Missing Figure 2a results for {name} {family} {role}")
            metric = f"recall@{k_for(modality)}"
            hs_paired = hs_trials[0][["seed", "run_dir", metric]].merge(
                hs_trials[1][["seed", "run_dir", metric]], on=["seed", "run_dir"],
                how="outer", validate="one_to_one", indicator=True,
                suffixes=("_synthetic", "_human"))
            if not hs_paired["_merge"].eq("both").all():
                raise ValueError(f"Synthetic/human checkpoints differ for {name} {family} {role}")
            if set(hs_paired["seed"]) != {42, 43, 44}:
                raise ValueError(f"Expected all three seeds for {name} {family} {role}")
            for _, trial in hs_paired.sort_values("seed").iterrows():
                hs_rows.append({"family": family, "group": role, "modality": modality,
                                "seed": int(trial["seed"]), "run_dir": trial["run_dir"],
                                "reference": False,
                                "recall_k": k_for(modality),
                                "synthetic_recall": trial[metric + "_synthetic"],
                                "human_matched_recall": trial[metric + "_human"]})
    hs_summary = pd.DataFrame(hs_rows)
    hs_summaries[name] = hs_summary
    hs_summary.to_csv(os.path.join(WORK_DIR, f"human_vs_synthetic_{name}_recall.csv"), index=False)

    hs_fig, hs_ax = plt.subplots(figsize=(7.5, 6 * 0.90))
    hs_ax.grid(False)
    # Fit the linear axes to every model, including the baseline CoSENT control.
    hs_xs = hs_summary["synthetic_recall"].to_numpy()
    hs_ys = hs_summary["human_matched_recall"].to_numpy()
    for axis in (hs_ax.xaxis, hs_ax.yaxis):
        axis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    shared_span = max(np.ptp(hs_xs), np.ptp(hs_ys))
    half_span = shared_span / 2 + max(0.03 * shared_span, 0.001)
    center_x = (hs_xs.min() + hs_xs.max()) / 2
    center_y = (hs_ys.min() + hs_ys.max()) / 2
    hs_plot_limits[name] = {"x": (center_x - half_span, center_x + half_span),
                            "y": (center_y - half_span, center_y + half_span)}
    hs_ax.set_xticks(nice_ticks(*hs_plot_limits[name]["x"]))
    hs_ax.set_yticks(nice_ticks(*hs_plot_limits[name]["y"]))
    hs_ax.set_xlim(*hs_plot_limits[name]["x"])
    hs_ax.set_ylim(*hs_plot_limits[name]["y"])
    hs_ax.set_aspect("equal", adjustable="box")
    hs_ax.set_autoscale_on(False)
    for family in FAMILY_ORDER:
        colour = family_colour[family]
        for role in roles_for(modality):
            marker = ROLE_MARKER[role]
            filled = ROLE_FILLED[role]
            hs_group = hs_summary[(hs_summary["family"] == family) & (hs_summary["group"] == role)]
            hs_ax.scatter(hs_group["synthetic_recall"], hs_group["human_matched_recall"],
                          marker=marker, s=ROLE_SIZE[role],
                          facecolor=colour if filled else "none",
                          edgecolor="white" if filled else colour,
                          linewidth=0.5 if filled else 1.2, zorder=3)
    # Fit all selected seed-level checkpoints available in this modality.
    # Baseline CoSENT participates in both the fit and the axis limits.
    fit_frame = hs_summary
    fit_xs = fit_frame["synthetic_recall"].to_numpy()
    fit_ys = fit_frame["human_matched_recall"].to_numpy()
    hs_fit = stats.linregress(fit_xs, fit_ys)
    hs_fit_x = np.linspace(min(hs_xs), max(hs_xs), 200)
    hs_fit_line, = hs_ax.plot(hs_fit_x, hs_fit.intercept + hs_fit.slope * hs_fit_x,
                             color="0.4", linestyle="--", linewidth=1.3, zorder=1,
                             label=fr"Linear fit ($R^2={hs_fit.rvalue ** 2:.3f}$)")
    hs_fit_rows.append({"modality": modality, "recall_k": k_for(modality),
                        "n_points": len(fit_frame), "slope": hs_fit.slope,
                        "intercept": hs_fit.intercept, "pearson_r": hs_fit.rvalue,
                        "r_squared": hs_fit.rvalue ** 2,
                        "spearman_rho": stats.spearmanr(fit_xs, fit_ys).statistic})
    hs_blank = Line2D([], [], linewidth=0, label="")
    # Query kinds are encoded by the axes; no training-strategy arrows are drawn.
    hs_ax.legend(
        handles=[Line2D([], [], color=family_colour[f], linewidth=3, label=FAMILY_LABEL[f])
                 for f in FAMILY_ORDER]
        + [hs_blank]
        + [Line2D([], [], color="0.35", marker=m, linewidth=0, markersize=np.sqrt(ROLE_SIZE[r]),
                  markerfacecolor="0.35" if ROLE_FILLED[r] else "none", label=GROUP_LEGEND_LABEL[r])
           for r, m in reversed(list(ROLE_MARKER.items())) if r in roles_for(modality)]
        + [hs_blank, hs_fit_line],
        loc="upper left", fontsize=8)
    hs_ax.set_xlabel(f"Synthetic {name} Recall@{k_for(modality)}")
    hs_ax.set_ylabel(f"Human (matched) {name} Recall@{k_for(modality)}")
    hs_fig.tight_layout()
    for ext in ("png", "pdf"):
        hs_fig.savefig(os.path.join(WORK_DIR, f"human_vs_synthetic_{name}_recall.{ext}"), dpi=180)
    hs_figures[name] = (hs_fig, hs_ax)
    plt.show()
    print(f"{name.title()}: {len(hs_summary)} points ({len(hs_summary) // 3} model/strategy groups × 3 seeds); "
          f"R²={hs_fit.rvalue ** 2:.4f}")
    display(hs_summary.set_index(["family", "group", "seed"]).round(4))

hs_fits = pd.DataFrame(hs_fit_rows)
hs_fits.to_csv(os.path.join(WORK_DIR, "human_vs_synthetic_recall_fits.csv"), index=False)
display(hs_fits.round(4))

# Compact paper panels use the same points and fits, with Figure 2a as the shared legend.
hs_paper_figures = {}
for name in ("text", "image"):
    frame = hs_summaries[name]
    fit = hs_fits[hs_fits["modality"] == frame["modality"].iloc[0]].iloc[0]
    paper_fig, paper_ax = plt.subplots(figsize=(2.3, 2.2))
    paper_ax.grid(False)
    for family in FAMILY_ORDER:
        for role in roles_for("text" if name == "text" else "multimodal"):
            marker = ROLE_MARKER[role]
            group = frame[(frame["family"] == family) & (frame["group"] == role)]
            filled = ROLE_FILLED[role]
            paper_ax.scatter(group["synthetic_recall"], group["human_matched_recall"],
                             marker=marker, s=ROLE_SIZE[role] * 0.3,
                             facecolor=family_colour[family] if filled else "none",
                             edgecolor="white" if filled else family_colour[family],
                             linewidth=0.3 if filled else 0.7, zorder=3)
    xs = frame["synthetic_recall"].to_numpy()
    ys = frame["human_matched_recall"].to_numpy()
    line_x = np.linspace(xs.min(), xs.max(), 200)
    paper_ax.plot(line_x, fit.intercept + fit.slope * line_x,
                  color="0.4", linestyle="--", linewidth=0.8, zorder=1)
    paper_ax.set_xticks(nice_ticks(*hs_plot_limits[name]["x"], target=6))
    paper_ax.set_yticks(nice_ticks(*hs_plot_limits[name]["y"], target=6))
    paper_ax.set_xlim(*hs_plot_limits[name]["x"])
    paper_ax.set_ylim(*hs_plot_limits[name]["y"])
    paper_ax.set_aspect("equal", adjustable="box")
    for axis in (paper_ax.xaxis, paper_ax.yaxis):
        axis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    paper_ax.tick_params(labelsize=7, length=3, pad=2)
    paper_ax.set_title(name.title(), fontsize=9, pad=4)
    paper_ax.set_xlabel(f"Synthetic Recall@{int(fit.recall_k)}", fontsize=8, labelpad=2)
    paper_ax.set_ylabel(f"Human Recall@{int(fit.recall_k)}", fontsize=8, labelpad=2)
    paper_ax.text(0.97, 0.04, fr"$R^2={fit.r_squared:.3f}$", transform=paper_ax.transAxes,
                  ha="right", va="bottom", fontsize=7,
                  bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=0.5))
    paper_fig.subplots_adjust(left=0.24, right=0.97, bottom=0.21, top=0.88)
    for ext in ("png", "pdf"):
        paper_fig.savefig(os.path.join(FIG_DIR, f"human_vs_synthetic_{name}_recall.{ext}"), dpi=220)
    hs_paper_figures[name] = (paper_fig, paper_ax)
    plt.close(paper_fig)

hs_table = hs_fits.set_index("modality")
hs_table_lines = [r"\begingroup", r"\scriptsize", r"\setlength{\tabcolsep}{2.5pt}",
                  r"\begin{tabular}{@{}lrr@{}}", r"\toprule", r"Measure & Text & Image \\",
                  r"\midrule"]
for label, column in ((r"Pearson $r$", "pearson_r"),
                      (r"Spearman $\rho$", "spearman_rho")):
    hs_table_lines.append(f"{label} & {hs_table.loc['text', column]:#.2g} & "
                          f"{hs_table.loc['multimodal', column]:#.2g}" + r" \\")
hs_table_lines += [r"\bottomrule", r"\end{tabular}", r"\endgroup"]
with open(os.path.join(FIG_DIR, "human_synthetic_correlations.tex"), "w") as handle:
    handle.write("\n".join(hs_table_lines) + "\n")


### 3b. Paired significance tests

Compare GOLD-mined and GOLD-reweighted with both baselines, and compare reweighting with GOLD-mined. Tests report training-seed sensitivity and Holm-adjusted two-sided p-values within each query kind and modality.


In [ ]:
# Paired permutation tests on the per-query recall behind the figure and table above
# (utils/significance.py). Each row is one difference the paper argues about, A - B, on the
# same (kind, family, group) slots the figure draws; a slot without a point is a pending row.
from utils.significance import compare, holm_adjust

COMPARISONS = [("ours-ungraded", "baseline"), ("ours-graded", "baseline"),
               ("ours-graded", "ours-ungraded"), ("ours-ungraded", "nv-mined"),
               ("ours-graded", "nv-mined")]
N_PERM = 10000
SIG_LEVEL = 0.05


def role_runs(pair, role, modality, kind):
    """{training seed: run_dir} behind one figure point, or None when the slot is empty."""
    trials = role_trials(pair, role, modality, kind)
    if trials is None:
        return None
    if trials["seed"].duplicated().any():
        raise ValueError(f"{role} {modality} {kind}: repeated seeds {sorted(trials['seed'])}")
    return dict(zip(trials["seed"], trials["run_dir"]))


sig_rows = []
for kind in PLOT_KINDS:
    subdir = "preds_human" if kind.startswith("human") else "preds"
    for modality in ("text", "multimodal"):
        for family in FAMILY_ORDER:
            pair = PROTOCOL[family]
            for role_a, role_b in COMPARISONS:
                row = {"kind": kind, "modality": modality, "family": family, "a": role_a, "b": role_b}
                row["status"] = "paired"
                runs_a = role_runs(pair, role_a, modality, kind)
                runs_b = role_runs(pair, role_b, modality, kind)
                if runs_a is not None and runs_b is not None:
                    res = compare(runs_a, runs_b, k_for(modality), subdir, n_perm=N_PERM)
                    row["n_seeds"] = len(res.pop("seed_pairings"))
                    row.update(res)
                sig_rows.append(row)
significance = pd.DataFrame(sig_rows)
# Holm adjustment over all completed comparisons within each query kind and modality.
tested = significance["p_two_sided"].notna()
significance.loc[tested, "p_two_sided_holm"] = (significance[tested]
                                                 .groupby(["kind", "modality"])["p_two_sided"]
                                                 .transform(holm_adjust))
significance.to_csv(os.path.join(WORK_DIR, "significance.csv"), index=False)
display(significance.round(4))

# A verdict that holds pooled but fails on some single seed pairing is reported, not hidden.
fragile = significance[tested & (significance["p_greater"] < SIG_LEVEL)
                       & (significance["p_greater_pairing_max"] >= SIG_LEVEL)]
if not fragile.empty:
    print(f"pooled p < {SIG_LEVEL} but some seed pairing is not:")
    display(fragile[["kind", "modality", "family", "a", "b", "delta", "p_greater",
                     "delta_pairing_min", "delta_pairing_max", "p_greater_pairing_max"]].round(4))

# The tabular main.tex \input{}s as Table tab:significance: A - B per cell with the one-sided
# p that A is better than B, bold where it is below SIG_LEVEL.
# The difference column reuses table 3's GROUP_LABEL vocabulary with the trailing noun and
# the "(ours)" tag dropped, so the two tables name the same groups the same way while the
# "A - B" labels stay inside the text block.
SHORT_GROUP = {role: label.replace(" negatives", "").replace(" (ours)", "")
               for role, label in GROUP_LABEL.items()}


def sig_cell(row):
    if pd.isna(row["delta"]):
        return r"\textcolor{red}{not paired}"
    p = row["p_greater"]
    p_text = "$p<.001$" if p < 0.001 else "$p=" + f"{p:.3f}".replace("0.", ".") + "$"
    text = f"{round(row['delta'], 3) + 0.0:+.3f}".replace("0.", ".")
    if p < SIG_LEVEL:
        text = r"\textbf{" + text + "}"
    return text + r"\,{\tiny " + p_text + "}"


sig_index = significance.set_index(["kind", "modality", "family", "a", "b"])
header = " & ".join(["Loss", r"$A$ vs.\ $B$"] + [f"{TEX_KIND[k].replace('(matched)', '').strip()} {'text' if m == 'text' else 'image'}"
                                           for k, m in TEX_COLUMNS])
# The "A - B" column is wide enough that the default 6pt tabcolsep overflows the text
# block by about 20pt; 4pt over the six column gaps recovers 24pt.
lines = [r"\scriptsize", r"\setlength{\tabcolsep}{4pt}",
         r"\begin{tabular}{llrrrr}", r"    \toprule",
         f"    {header} \\\\",
         "    & & " + " & ".join(f"(Recall@{k_for(m)})" for _, m in TEX_COLUMNS) + r" \\",
         r"    \midrule"]
for family in FAMILY_ORDER:
    for position, (role_a, role_b) in enumerate(COMPARISONS):
        name = TEX_FAMILY[family] if position == 0 else ""
        label = (r"\begin{tabular}[c]{@{}l@{}}" + SHORT_GROUP[role_a]
                 + r"\\vs.\ " + SHORT_GROUP[role_b] + r"\end{tabular}")
        cells = [(r"\red{" + sig_cell(sig_index.loc[(kind, modality, family, role_a, role_b)]) + "}"
                  if kind.startswith("human") or modality == "text" else sig_cell(sig_index.loc[(kind, modality, family, role_a, role_b)]))
                 for kind, modality in TEX_COLUMNS]
        lines.append("    " + " & ".join([name, label] + cells) + r" \\")
    if family != FAMILY_ORDER[-1]:
        lines.append(r"    \midrule")
lines += [r"    \bottomrule", r"\end{tabular}"]
sig_tex = os.path.join(FIG_DIR, "significance_table.tex")
with open(sig_tex, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"wrote {sig_tex}")


## 4. Win rate: positive vs. hard negative

Pairwise accuracy: for each (query, positive, hard negative) triple the model **wins** when it
scores the positive above the negative. `sim_pos`/`sim_neg` are already stored per row in every
run's `preds/triplets.jsonl`, so this is a read of existing preds, not a re-encode.

Easy negatives (`negative_example_source == "random"`) are excluded -- they are the shared random
distractors, not the mined hard negative this experiment is about. Ties count as losses; the
tie count is printed below so it stays visible rather than assumed negligible.

Same protocol pairs and rephrased main-grid rows as section 3, with the same group
ordering by mean Recall over `RANK_KINDS`. Human-written queries have no verified hard
negatives and are evaluated with retrieval recall in section 3.

Each family panel also carries the section 3 extra groups: the
`baseline` control and `nv-mined` baseline (the ungraded style on the baseline-negative sibling,
`baseline_row`, and on the NV-mined negatives, `nv_row`). The y-axis starts at chance (0.5), so bar height is the margin above
a coin flip.

Win rates are joined to the section 3 trials on the full run identity (`negs`, `mining`,
`order` and `seed` included), so a slot's mean is over its own seeds only. The table below is
indexed by `(modality, family, style, negs)`: `labeled` rows are the protocol pair, `baseline`
and `mined` the control and the NV baseline on the ungraded style. Rows are ordered by modality, then family in `PROTOCOL` order, then by `rephrased-ic`
win rate, best first.


In [ ]:
from utils.distance_labels import EASY_NEGATIVE_SOURCE


def win_rate(run_dir, preds_subdir="preds"):
    """Fraction of hard-negative pairs whose positive scores above the negative."""
    with open(os.path.join(run_dir, preds_subdir, "triplets.jsonl"), encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    hard = [r for r in rows if r["negative_example_source"] != EASY_NEGATIVE_SOURCE]
    return {"win_rate": sum(r["sim_pos"] > r["sim_neg"] for r in hard) / len(hard),
            "n_pairs": len(hard),
            "n_ties": sum(r["sim_pos"] == r["sim_neg"] for r in hard)}


# One row per trial, keyed like `results` so a slot's trials join their own win rates and not
# those of the same style trained on baseline or mined negatives, or of another seed.
WIN_KEYS = ["modality", "style", "query_kind", "train_kind", "rephrase", "V", "easy",
            "negs", "mining", "order", "seed"]


def win_row(r, preds_subdir):
    return {"modality": r.modality, "style": r.style, "query_kind": scored_kind(r, preds_subdir),
            "train_kind": r.query_kind, "rephrase": r.rephrase, "V": r.V, "easy": r.easy,
            "negs": r.negs, "mining": r.mining, "order": r.order, "seed": r.seed,
            **win_rate(r.run_dir, preds_subdir)}


wins = pd.DataFrame([win_row(r, "preds") for r in usable.itertuples()])
print(f"{len(wins)} conditions | {wins['n_pairs'].sum():,} hard-negative pairs | "
      f"{wins['n_ties'].sum():,} exact ties (counted as losses)")


In [ ]:
# Slots and trials come from section 3: the protocol pair from `main` (with its `missing` /
# `incomplete` notes) and the family's extra groups from baseline_row() and nv_row(), so the
# groups and their ordering cannot drift from that figure.
def slot_wins(trials):
    """(mean win rate, n trials) over a slot's trials."""
    keys = trials[WIN_KEYS].assign(query_kind=trials["query_kind"].astype(str))
    hit = keys.merge(wins, on=WIN_KEYS, how="left")
    if len(hit) != len(trials) or hit["win_rate"].isna().any():
        raise ValueError(f"win rates do not join one-to-one onto trials:\n{hit}")
    return float(hit["win_rate"].mean()), len(hit)


win_rows = []
for family, pair in PROTOCOL.items():
    ungraded, graded = pair[0], pair[-1]
    # extra group -> (its section 3 row function, the style it trains, its negs label)
    extra = {BASELINE: (baseline_row, ungraded, "baseline"),
             NV_MINED: (nv_row, ungraded, "mined")}
    slots, fam_missing, fam_incomplete = {}, dict(missing), dict(incomplete)
    for modality in ["text", "multimodal"]:
        for kind in RANK_KINDS:
            for style in pair:
                trials = main[(main["modality"] == modality) & (main["style"] == style)
                              & (main["query_kind"] == kind)]
                if not trials.empty:
                    slots[modality, style, kind] = trials
            for group, (row_of, style, _) in extra.items():
                trials, note = row_of(modality, kind, style)
                if trials is None:
                    fam_missing[modality, group, kind] = note
                    continue
                slots[modality, group, kind] = trials
                if note:
                    fam_incomplete[modality, group, kind] = note

    def win_of(modality, style, kind):
        return slot_wins(slots[modality, style, kind])[0] if (modality, style, kind) in slots else None

    def slot_recall(modality, style, kind):
        """Section 3's ranking value for these groups, so the bar order matches that figure."""
        return (trial_stats(slots[modality, style, kind], k_for(modality))[0]
                if (modality, style, kind) in slots else None)

    for (modality, group, kind), trials in slots.items():
        mean, n = slot_wins(trials)
        style, negs = (extra[group][1], extra[group][2]) if group in extra else (group, "labeled")
        win_rows.append({"family": family, "modality": modality, "style": style, "negs": BASELINE_NEGS[modality] if group == BASELINE else negs,
                         "query_kind": kind, "win_rate": mean, "n": n})

    fig, axes = plt.subplots(1, 2, figsize=(max(11, 3.2 * (len(pair) + len(extra))), 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        groups, kinds = ordered_groups(modality, list(pair) + baseline_groups(modality) + [NV_MINED], slot_recall)
        kinds = [kind for kind in kinds if kind in RANK_KINDS]
        width = 0.8 / len(kinds)
        for j, k in enumerate(kinds):
            xs = [cell_x(i, j, width, kinds) for i in range(len(groups))]
            ys = [win_of(modality, s, k) for s in groups]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    draw_missing(ax, i, j, width, kinds, fam_missing[modality, s, k])
                elif (modality, s, k) in fam_incomplete:
                    label_cell(ax, xs[i], fam_incomplete[modality, s, k])
        # Floor at chance rather than 0: a win rate below 0.5 would be worse than a coin
        # flip, so bar height reads as the margin above chance.
        ax.set_ylim(bottom=0.5)
        if not any(win_of(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0.5, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{'Baseline' if s == BASELINE else s}\n({ROLE[s]})" for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel("win rate vs hard negative")
    axes[0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{family}: baseline vs ungraded vs graded vs nv-mined -- win rate", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(WORK_DIR, f"winrate_{family}.png"), dpi=150)
    plt.show()

# Rows: modality, then family in PROTOCOL order, then best rephrased-ic win rate first.
win_table = (pd.DataFrame(win_rows)
             .pivot_table(index=["modality", "family", "style", "negs"], columns="query_kind",
                          values="win_rate", observed=True)
             .reset_index())
win_table["modality"] = pd.Categorical(win_table["modality"], ["text", "multimodal"])
win_table["family"] = pd.Categorical(win_table["family"], list(PROTOCOL))
win_table = (win_table.sort_values(["modality", "family", RANK_KINDS[0]], ascending=[True, True, False])
             .set_index(["modality", "family", "style", "negs"]))
display(win_table.round(4))


### 4b. V versus hard-negative win rate

Use **existing checkpoints and saved validation predictions** for the current graded
InfoNCE, SigLIP (`siglip-v3`), and Margin-MSE losses on our labeled negatives.
CoSENT has no V parameter. Queries are rephrased; no human-study data are used.
Validation predictions provide the existing V sweeps; selected test results are not mixed
into these curves. No training or inference is launched.

Each curve holds the model, dataset, rephrasing variant, easy-negative distance, and
validation split fixed. In-context and plain rephrasing are kept separate, and Margin-MSE
has separate curves for each easy-negative setting. Evaluation pairs must match along a
curve. Text and image use separate side-by-side plots with a shared win-rate scale.
Each plot contains all loss/settings curves; colour identifies the loss and markers
distinguish easy-negative settings. V is on a base-2 log axis, with actual V values marked.

Win rate is the fraction of hard-negative pairs with `sim_pos > sim_neg`; ties are losses.
Points average the available training trials equally. **n** in the legend is the number
of checkpoint trials at each V, while **N** is the number of evaluated hard-negative
pairs **per trial**. Error bars span the observed minimum–maximum across trials; at n=1
there is no estimated between-trial variation. Sparse coverage describes the existing runs,
not a controlled multi-trial sweep. Missing or stale predictions are listed as skipped.

Writes `analysis/figs/v_winrate.{png,pdf}`, `v_winrate_runs.csv`, `v_winrate_summary.csv`,
and `v_winrate_skipped.csv`.


In [ ]:
from utils.paper_analysis import staleness

# Current graded losses with a V parameter; CoSENT has none.
V_WIN_STYLES = {pair[-1]: family for family, pair in PROTOCOL.items() if pair[-1] in SWEEP_GRID}
v_win_candidates = discover_runs(MODELS_ROOT, note=NOTE)
v_win_candidates = v_win_candidates[
    (v_win_candidates["query_kind"] == ANALYSIS_QUERY_KIND)
    & v_win_candidates["style"].isin(V_WIN_STYLES)
    & v_win_candidates["V"].notna()
    & (v_win_candidates["negs"] == "labeled")
    & (v_win_candidates["order"] == "")].copy()
v_win_candidates["easy"] = v_win_candidates["easy"].fillna(20).astype(int)

v_win_series = ["modality", "family", "style", "model_short", "rephrase", "easy", "dataset", "split_seed"]
v_win_rows, v_win_skipped, v_win_reference = [], [], {}
for run in v_win_candidates.itertuples():
    preds = os.path.join(run.run_dir, "preds_val")
    if not all(os.path.isfile(os.path.join(preds, name)) for name in ("meta.json", "triplets.jsonl")):
        v_win_skipped.append({"run_dir": run.run_dir, "reason": "no saved validation triplets"})
        continue
    reason = staleness(run.run_dir, preds_subdir="preds_val")
    if reason:
        v_win_skipped.append({"run_dir": run.run_dir, "reason": reason})
        continue
    with open(os.path.join(preds, "meta.json")) as handle:
        meta = json.load(handle)
    if meta["split"] != "validation" or meta["args"]["query_kind"] != ANALYSIS_QUERY_KIND or "_rephrased-in-context" not in meta["args"]["dataset"]:
        raise ValueError(f"{run.run_dir}: expected rephrased validation predictions")
    row = {key: getattr(run, key) for key in ("modality", "style", "model_short", "rephrase", "easy", "V", "seed", "run_dir")}
    row.update(family=V_WIN_STYLES[run.style], dataset=meta["args"]["dataset"],
               split_seed=meta["args"]["split_seed"])
    # Confirm that each curve changes V/training trial, not its evaluation pairs.
    with open(os.path.join(preds, "triplets.jsonl")) as handle:
        hard = [t for line in handle if line.strip()
                for t in [json.loads(line)] if t["negative_example_source"] != EASY_NEGATIVE_SOURCE]
    if not hard:
        v_win_skipped.append({"run_dir": run.run_dir, "reason": "no hard-negative pairs"})
        continue
    if not np.isfinite([[t["sim_pos"], t["sim_neg"]] for t in hard]).all():
        raise ValueError(f"{run.run_dir}: non-finite pair similarities")
    signature = tuple(sorted((t["query_id"], t["positive_corpus_id"], t["negative_corpus_id"]) for t in hard))
    series = tuple(row[key] for key in v_win_series)
    if series in v_win_reference and signature != v_win_reference[series]:
        raise ValueError(f"{run.run_dir}: evaluation pairs differ within the V curve")
    v_win_reference[series] = signature
    row.update(win_rate(run.run_dir, preds_subdir="preds_val"))
    v_win_rows.append(row)

if not v_win_rows:
    raise ValueError("No current checkpoints have saved validation predictions for the V analysis")
v_win_runs = pd.DataFrame(v_win_rows)
if v_win_runs.duplicated(v_win_series + ["V", "seed"]).any():
    raise ValueError("Repeated training trials within a V setting; do not count them as independent runs")
v_win_summary = (v_win_runs.groupby(v_win_series + ["V"], as_index=False, dropna=False)
                 .agg(win_rate=("win_rate", "mean"), win_min=("win_rate", "min"),
                      win_max=("win_rate", "max"), n=("seed", "nunique"),
                      n_pairs=("n_pairs", "first"), n_ties=("n_ties", "sum")))
v_win_runs.to_csv(os.path.join(WORK_DIR, "v_winrate_runs.csv"), index=False)
v_win_summary.to_csv(os.path.join(WORK_DIR, "v_winrate_summary.csv"), index=False)
v_win_skipped = pd.DataFrame(v_win_skipped, columns=["run_dir", "reason"])
v_win_skipped.to_csv(os.path.join(WORK_DIR, "v_winrate_skipped.csv"), index=False)
print(f"{len(v_win_runs)} existing checkpoints; {len(v_win_summary)} V/settings; "
      f"{len(v_win_skipped)} checkpoints without usable saved validation triplets")
print("n = training trials per point; N = hard-negative pairs per trial. Ties count as losses.")
display(v_win_summary[["modality", "family", "rephrase", "easy", "V", "win_rate", "n", "n_pairs", "n_ties"]]
        .rename(columns={"n_pairs": "N_pairs_per_trial"}).round(4))
if not v_win_skipped.empty:
    display(v_win_skipped.groupby("reason").size().rename("checkpoints").to_frame())

v_win_families = [family for family in FAMILY_ORDER if family in set(v_win_summary["family"])]
v_win_levels = sorted(v_win_summary["V"].unique())
v_win_colours = dict(zip(v_win_families, sns.color_palette("colorblind", len(v_win_families))))
v_win_markers = {10: "o", 20: "s", 40: "^"}
v_win_fig, v_win_axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True, layout="constrained")
v_win_bottom = min(0.5, max(0, float(v_win_summary["win_min"].min()) - 0.05))
for ax, modality, modality_label in zip(v_win_axes, ("text", "multimodal"), ("Text", "Image")):
    for family in v_win_families:
        sub = v_win_summary[(v_win_summary["family"] == family) & (v_win_summary["modality"] == modality)]
        for _, curve in sub.groupby(v_win_series, dropna=False, sort=True):
            curve = curve.sort_values("V")
            first = curve.iloc[0]
            variant = "in-context" if first.rephrase == "in-context" else "rephrased"
            n_label = (f"n={int(first.n)}" if curve["n"].nunique() == 1
                       else "n by V: " + ", ".join(f"{p.V:g}:{p.n}" for p in curve.itertuples()))
            label = (f"{FAMILY_LABEL[family]} · {variant}, easy={int(first.easy)}"
                     f" · {n_label}, N={int(first.n_pairs):,}")
            low = curve["win_rate"] - curve["win_min"]
            high = curve["win_max"] - curve["win_rate"]
            ax.errorbar(curve["V"], curve["win_rate"], yerr=np.vstack([low, high]),
                        color=v_win_colours[family], linestyle="-",
                        marker=v_win_markers.get(int(first.easy), "D"),
                        markersize=5, capsize=3, linewidth=1.6, label=label)
    ax.set_title(modality_label)
    ax.set_xlabel("V")
    ax.set_ylabel("Win rate vs. hard negative" if modality == "text" else "")
    ax.set_ylim(v_win_bottom, 1.03)
    ax.set_xscale("log", base=2)
    ax.set_xticks(v_win_levels, labels=[f"{v:g}" for v in v_win_levels])
    ax.set_xlim(min(v_win_levels) / 1.25, max(v_win_levels) * 1.25)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), fontsize=8,
              title="n: trials per V · N: pairs per trial", title_fontsize=9)
v_win_fig.suptitle("V and hard-negative win rate · existing validation predictions", fontsize=14)
for ext in ("png", "pdf"):
    v_win_fig.savefig(os.path.join(WORK_DIR, f"v_winrate.{ext}"), dpi=180, bbox_inches="tight")
plt.show()


### 4c. Text versus image win rate (Figure 2 style)

The same selected models, family colours, strategy markers, filled/unfilled symbols,
legend labels, and transition arrows as Figure 2, with **text win rate on x** and
**image win rate on y**. Each point averages the available training trials for the
in-context rephrased test queries; human-study queries are excluded.

Uses Figure 2's linear axes, two-decimal ticks, and 3% padding, including all Baseline points. Limits are fitted to win rates: Figure 2's numeric recall limits would hide
the image win-rate points. Any clipped or missing points are reported below the plot.
The summary table records the number of trials for each modality.

Writes `analysis/figs/text_vs_image_winrate.{png,pdf,csv}`.



In [ ]:
# Figure 2's selected models and visual encodings, scored by hard-negative win rate.
WIN_SCATTER_KIND = RANK_KINDS[0]
win_scatter_points, win_scatter_rows, win_scatter_missing = {}, [], []
for family in FAMILY_ORDER:
    for role in ROLE_MARKER:
        means = {}
        for modality in ("text", "multimodal"):
            trials = role_trials(PROTOCOL[family], role, modality, WIN_SCATTER_KIND)
            if trials is None or trials.empty:
                win_scatter_missing.append((family, role, modality))
                continue
            mean, n = slot_wins(trials)
            means[modality] = mean
            win_scatter_rows.append({"family": family, "role": role, "modality": modality,
                                     "query_kind": WIN_SCATTER_KIND, "win_rate": mean, "n": n,
                                     "source_group": role, "reference": False})
        if set(means) == {"text", "multimodal"}:
            win_scatter_points[family, role] = (means["text"], means["multimodal"])

win_scatter_summary = pd.DataFrame(win_scatter_rows)
win_scatter_summary.to_csv(os.path.join(WORK_DIR, "text_vs_image_winrate.csv"), index=False)
if not win_scatter_points:
    raise ValueError("No selected models have win rates for both text and image")

win_scatter_fig, win_scatter_ax = plt.subplots(figsize=(7.5, 6))
win_scatter_ax.grid(False)
# Match Figure 2's linear scales, tick formatting, padding, and outlier policy.
# Limits follow win rates rather than recall, whose image range would hide these points.
win_scale_points = [point for key, point in win_scatter_points.items() if key not in SCALE_EXCLUDE]
win_xs, win_ys = zip(*(win_scale_points or list(win_scatter_points.values())))
for axis in (win_scatter_ax.xaxis, win_scatter_ax.yaxis):
    axis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
win_scatter_ax.set_xticks(nice_ticks(min(win_xs), max(win_xs)))
win_scatter_ax.set_yticks(nice_ticks(min(win_ys), max(win_ys)))
win_pad_x = max(0.03 * (max(win_xs) - min(win_xs)), 0.001)
win_pad_y = max(0.03 * (max(win_ys) - min(win_ys)), 0.001)
win_scatter_ax.set_xlim(min(win_xs) - win_pad_x, max(win_xs) + win_pad_x)
win_scatter_ax.set_ylim(min(win_ys) - win_pad_y, max(win_ys) + win_pad_y)
win_scatter_ax.set_autoscale_on(False)

for family in FAMILY_ORDER:
    colour = family_colour[family]
    for tail_role, head_role in role_edges_for("text"):
        if (family, tail_role) in win_scatter_points and (family, head_role) in win_scatter_points:
            win_scatter_ax.add_patch(FancyArrowPatch(
                win_scatter_points[family, tail_role], win_scatter_points[family, head_role],
                arrowstyle="->", mutation_scale=12, color=colour,
                linestyle=KIND_LINESTYLE[WIN_SCATTER_KIND], linewidth=1.1,
                alpha=0.55, shrinkA=7, shrinkB=9, zorder=2))
    for role, marker in ROLE_MARKER.items():
        if (family, role) in win_scatter_points:
            x, y = win_scatter_points[family, role]
            filled = ROLE_FILLED[role]
            win_scatter_ax.scatter([x], [y], marker=marker, s=ROLE_SIZE[role],
                                   facecolor=colour if filled else "none",
                                   edgecolor="white" if filled else colour,
                                   linewidth=0.5 if filled else 1.2, zorder=3)

win_legend_blank = Line2D([], [], linewidth=0, label="")
win_scatter_ax.legend(
    handles=[Line2D([], [], color=family_colour[f], linewidth=3, label=FAMILY_LABEL[f])
             for f in FAMILY_ORDER]
    + [win_legend_blank]
    + [Line2D([], [], color="0.35", marker=m, linewidth=0, markersize=np.sqrt(ROLE_SIZE[r]),
              markerfacecolor="0.35" if ROLE_FILLED[r] else "none", label=PAIRED_GROUP_LABEL[r])
       for r, m in reversed(list(ROLE_MARKER.items()))]
    + [win_legend_blank]
    + [Line2D([], [], color="0.35", linestyle=KIND_LINESTYLE[WIN_SCATTER_KIND],
              linewidth=1.4, label=KIND_LABEL[WIN_SCATTER_KIND])],
    loc="upper left", fontsize=8)
win_scatter_ax.set_xlabel("Text win rate vs. hard negative")
win_scatter_ax.set_ylabel("Image win rate vs. hard negative")
win_scatter_fig.tight_layout()
for ext in ("png", "pdf"):
    win_scatter_fig.savefig(os.path.join(WORK_DIR, f"text_vs_image_winrate.{ext}"), dpi=180)
plt.show()

print(f"{len(win_scatter_points)} paired text/image points; "
      f"trials per modality: {sorted(win_scatter_summary['n'].unique())}")
win_xlim, win_ylim = win_scatter_ax.get_xlim(), win_scatter_ax.get_ylim()
win_clipped = [f"{f}/{r}" for (f, r), (x, y) in win_scatter_points.items()
               if not (win_xlim[0] <= x <= win_xlim[1] and win_ylim[0] <= y <= win_ylim[1])]
if win_clipped:
    print("Outside the axes under Figure 2's scale policy: " + ", ".join(win_clipped))
if win_scatter_missing:
    print("Missing model/modality slots:", win_scatter_missing)
display(win_scatter_summary.pivot(index=["family", "role"], columns="modality",
                                 values=["win_rate", "n"]).round(4))


### 4d. Recall versus win rate by loss and modality

The paper shows eight panels: text in the top row, images in the bottom row, and one column per loss family. Each panel places hard-negative win rate on the x-axis and recall on the y-axis, with independently fitted limits and dotted grids on both axes. Both baseline versions in text and all four image strategies and all points are included, averaging three trials. Arrows connect Baseline to each mining strategy, then GOLD mining to reweighting.


In [ ]:
# All four training strategies, evaluated on the same in-context rephrased test split.
WR_KIND = RANK_KINDS[0]
WR_ROLES = tuple(ROLE_MARKER)
wr_summaries, wr_points = {}, {}
for modality, name in (("text", "text"), ("multimodal", "image")):
    rows, points = [], {}
    for family in FAMILY_ORDER:
        for role in roles_for(modality):
            trials = role_trials(PROTOCOL[family], role, modality, WR_KIND)
            if trials is None or trials.empty:
                raise ValueError(f"Missing {modality} {family} {role} test results")
            if not (trials["rephrase"].eq("in-context").all()
                    and trials["train_kind"].eq("rephrased").all()
                    and trials["query_kind"].eq("rephrased-ic").all()):
                raise ValueError("These plots require rephrased-in-context training and test queries")
            mean_win, n = slot_wins(trials)
            mean_recall = float(trials[f"recall@{k_for(modality)}"].mean())
            points[family, role] = (mean_recall, mean_win)
            rows.append({"family": family, "role": role, "query_kind": WR_KIND,
                         "recall_k": k_for(modality), "recall": mean_recall,
                         "win_rate": mean_win, "n": n,
                         "reference": False})
    wr_summaries[name] = pd.DataFrame(rows)
    wr_points[name] = points
    wr_summaries[name].to_csv(os.path.join(WORK_DIR, f"{name}_winrate_vs_recall.csv"), index=False)

wr_family_handles = [Line2D([], [], color=family_colour[f], linewidth=3, label=FAMILY_LABEL[f])
                     for f in FAMILY_ORDER]
wr_role_handles = [Line2D([], [], color="0.35", marker=ROLE_MARKER[r], linewidth=0,
                         markersize=np.sqrt(ROLE_SIZE[r]), markerfacecolor="0.35" if ROLE_FILLED[r] else "none",
                         label=GROUP_LEGEND_LABEL[r].replace(" (ours)", "")) for r in WR_ROLES]


def draw_recall_winrate(ax, name, compact=False, family=None, shared_spans=None):
    """Shared Figure 3 colours and markers."""
    families = FAMILY_ORDER if family is None else [family]
    points = {(f, role): (win_rate, recall)
              for (f, role), (recall, win_rate) in wr_points[name].items() if f in families}
    ax.set_axisbelow(True)
    ax.grid(True, axis="both", linestyle=":", linewidth=0.6, alpha=0.7)
    xs, ys = zip(*points.values())
    for axis in (ax.xaxis, ax.yaxis):
        if family is None:
            axis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        else:
            axis.set_major_locator(mticker.MaxNLocator(nbins=3, min_n_ticks=3))
            axis.set_major_formatter(mticker.ScalarFormatter(useOffset=False))
    if family is None:
        ax.set_xticks(nice_ticks(min(xs), max(xs), target=5 if compact else 8))
        ax.set_yticks(nice_ticks(min(ys), max(ys), target=5 if compact else 8))
    if shared_spans is None:
        pad_x = max(0.05 * (max(xs) - min(xs)), 0.001)
        pad_y = max(0.05 * (max(ys) - min(ys)), 0.001)
        ax.set_xlim(min(xs) - pad_x, max(xs) + pad_x)
        ax.set_ylim(min(ys) - pad_y, max(ys) + pad_y)
    else:
        center_x, center_y = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
        half_x = 0.55 * max(shared_spans[0], 0.002)
        half_y = 0.55 * max(shared_spans[1], 0.002)
        ax.set_xlim(center_x - half_x, center_x + half_x)
        ax.set_ylim(center_y - half_y, center_y + half_y)
    ax.set_autoscale_on(False)
    for family in families:
        colour = family_colour[family]
        for role in roles_for("text" if name == "text" else "multimodal"):
            ax.scatter(*points[family, role], marker=ROLE_MARKER[role],
                       s=ROLE_SIZE[role] * (0.65 if compact else 1),
                       facecolor=colour if ROLE_FILLED[role] else "none",
                       edgecolor="white" if ROLE_FILLED[role] else colour,
                       linewidth=0.5 if ROLE_FILLED[role] else 1.2, zorder=3)
    modality = "text" if name == "text" else "multimodal"
    ax.set_xlabel("Win rate vs. hard negative", fontsize=9 if compact else 12)
    ax.set_ylabel(f"Recall@{k_for(modality)}", fontsize=9 if compact else 12)
    ax.tick_params(labelsize=8 if compact else 11)
    ax.set_title(name.title(), fontsize=10 if compact else 12)


# Keep the all-loss comparisons as standalone analysis exports.
wr_standalone = {}
for name in ("text", "image"):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    draw_recall_winrate(ax, name)
    ax.legend(handles=wr_family_handles + [h for r, h in zip(WR_ROLES, wr_role_handles)
                                          if r in roles_for("text" if name == "text" else "multimodal")],
              loc="best", fontsize=8)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(WORK_DIR, f"{name}_winrate_vs_recall.{ext}"), dpi=180)
    wr_standalone[name] = (fig, ax)
    plt.close(fig)

# Figure 4a–h: use one x span and one y span per modality, centered on each loss's points.
wr_panel_spans = {}
for name in ("text", "image"):
    spans = []
    for family in FAMILY_ORDER:
        points = [(win_rate, recall) for (loss, _), (recall, win_rate) in wr_points[name].items()
                  if loss == family]
        xs, ys = zip(*points)
        spans.append((max(xs) - min(xs), max(ys) - min(ys)))
    wr_panel_spans[name] = (max(span[0] for span in spans), max(span[1] for span in spans))

wr_loss_fig, wr_loss_axes = plt.subplots(2, 4, figsize=(10.5, 4.8), layout="constrained")
for row, name in enumerate(("text", "image")):
    for column, family in enumerate(FAMILY_ORDER):
        ax = wr_loss_axes[row, column]
        draw_recall_winrate(ax, name, compact=True, family=family,
                            shared_spans=wr_panel_spans[name])
        ax.set_title(f"({chr(ord('a') + row * 4 + column)}) {FAMILY_LABEL[family]}", fontsize=10)
        ax.set_ylabel(f"{name.title()}\nRecall@{K_TEXT if name == 'text' else K_IMAGE}" if column == 0 else "", fontsize=9)
for ext in ("png", "pdf"):
    wr_loss_fig.savefig(os.path.join(FIG_DIR, f"recall_vs_winrate.{ext}"), dpi=220)
plt.show()

# Retain the named data/axes for inspection alongside the earlier notebook comparisons.
text_wr_summary, image_wr_summary = wr_summaries["text"], wr_summaries["image"]
text_wr_points, image_wr_points = wr_points["text"], wr_points["image"]
text_wr_fig, text_wr_ax = wr_standalone["text"]
image_wr_fig, image_wr_ax = wr_standalone["image"]
for name in ("text", "image"):
    print(f"{name.title()}: {len(wr_points[name])} points; "
          f"trials per point: {sorted(wr_summaries[name]['n'].unique())}")
    display(wr_summaries[name].set_index(["family", "role"]).round(4))


## 5. Full table

Every rephrased-model condition, every cutoff, including its separate human-study evaluation.
Also written to analysis/figs/all_results.csv.

In [ ]:
table = (results.sort_values(["modality", "query_kind", f"recall@{K_MAIN}"],
                           ascending=[True, True, False])
         .reset_index(drop=True))
table.to_csv(os.path.join(WORK_DIR, "all_results.csv"), index=False)
display(table.round(4))


## 6. Accuracy by positive and negative condition counts

Only the **ours-graded** models from Figure 2: InfoNCE, SigLIP, Margin-MSE, and CoSENT,
on their **rephrased test queries**, using every seed in the figure. Accuracy uses the
paper's retrieval metric: **Recall@5 for text**, **Recall@20 for images**.

Counts come from the **original selected attribute lists**, before rephrasing:
positive = `selected_pos_features` + `selected_common_features`; negative =
`selected_neg_features` + `selected_neither_features`. Image datasets retain these lists.
Text counts come from the hashed BM25 generation journal, joined to each scored
`rephrased_query` and canonical `nl_query`. This also handles
commas inside individual attributes correctly. Only queries in saved test predictions
contribute; duplicate hard/easy-negative rows are counted once.

Each heatmap has positive count on x and negative count on y, both limited to 0–10. Cells show the mean recall
across seeds and `n`, the number of distinct queries (not multiplied by seeds). Empty
bins are grey, and all panels share a 0–1 colour scale. The per-seed CSV also preserves
seed variation. Figures and tables are saved under `analysis/figs/ours_graded_condition_*`.


In [ ]:
from utils.condition_counts import original_condition_counts

CONDITION_KIND = PLOT_KINDS[0]
CONDITION_ROLE = "ours-graded"


condition_frames, condition_run_rows = [], []
for modality in ("text", "multimodal"):
    reference, count_lookup, dataset_path = None, None, None
    k = k_for(modality)
    for family in FAMILY_ORDER:
        trials = role_trials(PROTOCOL[family], CONDITION_ROLE, modality, CONDITION_KIND)
        if trials is None or trials.empty or trials["seed"].duplicated().any():
            raise ValueError(f"{modality}/{family}: missing trials or repeated seeds")
        declared = main_grid[(main_grid["modality"] == modality)
                             & (main_grid["style"] == PROTOCOL[family][-1])
                             & (main_grid["query_kind"] == train_kind_of(CONDITION_KIND))
                             & (main_grid["rephrase"] == rephrase_of(CONDITION_KIND))]
        if set(trials["seed"]) != set(declared["seed"]):
            raise ValueError(f"{modality}/{family}: not all declared seeds have predictions")
        for run in trials.sort_values("seed").itertuples():
            preds = os.path.join(run.run_dir, "preds")
            with open(os.path.join(preds, "meta.json")) as handle:
                meta = json.load(handle)
            if meta["split"] != "test" or meta["args"]["query_kind"] != "rephrased":
                raise ValueError(f"{run.run_dir}: expected rephrased test predictions")
            with open(os.path.join(preds, "queries.jsonl"), encoding="utf-8") as handle:
                queries = [json.loads(line) for line in handle if line.strip()]
            signature = {q["query_id"]: (q["query"], tuple(sorted(q["positive_corpus_ids"])))
                         for q in queries}
            if len(signature) != len(queries) or len(queries) != meta["n_queries"]:
                raise ValueError(f"{run.run_dir}: duplicate or missing queries")
            if reference is None:
                reference = signature
                dataset_path = meta["args"]["dataset"]
                count_lookup = original_condition_counts(dataset_path, {q["query"] for q in queries})
            if signature != reference or meta["args"]["dataset"] != dataset_path:
                raise ValueError(f"{run.run_dir}: different test queries, positives, or dataset")
            rows = []
            for q in queries:
                positives = set(q["positive_corpus_ids"])
                if not positives or len(q["top_k"]) < k:
                    raise ValueError(f"{run.run_dir}: missing positives or fewer than {k} predictions")
                hits = positives.intersection(int(cid) for cid, _ in q["top_k"][:k])
                n_pos, n_neg = count_lookup[q["query"]]
                rows.append({"query_id": q["query_id"], "n_positive": n_pos, "n_negative": n_neg,
                             "recall": len(hits) / len(positives)})
            frame = pd.DataFrame(rows)
            np.testing.assert_allclose(frame["recall"].mean(), meta["metrics"][f"recall@{k}"], atol=1e-12)
            condition_frames.append(frame.assign(modality=modality, family=family,
                                                  style=run.style, seed=run.seed, k=k))
            condition_run_rows.append({"modality": modality, "family": family, "style": run.style,
                                       "seed": run.seed, "k": k, "n_queries": len(frame),
                                       "recall": frame["recall"].mean(), "run_dir": run.run_dir})
    print(f"{modality}: matched all {len(reference):,} test queries to original feature lists")

condition_query_scores = pd.concat(condition_frames, ignore_index=True)
condition_runs = pd.DataFrame(condition_run_rows)
display(condition_runs[["modality", "family", "style", "seed", "k", "n_queries", "recall"]].round(4))


In [ ]:
# Average queries within each seed and count bin, then average seeds with equal weight.
condition_keys = ["modality", "family", "style", "k", "n_positive", "n_negative"]
condition_seed_cells = (condition_query_scores.groupby(condition_keys + ["seed"], as_index=False)
                        .agg(recall=("recall", "mean"), n_queries=("query_id", "size")))
condition_cells = (condition_seed_cells.groupby(condition_keys, as_index=False)
                   .agg(recall=("recall", "mean"), seed_min=("recall", "min"),
                        seed_max=("recall", "max"), seed_std=("recall", "std"),
                        n_seeds=("seed", "nunique"), n_queries=("n_queries", "first")))
# n_queries is the number of distinct test queries in the bin, not multiplied by seeds.
condition_cells.to_csv(os.path.join(WORK_DIR, "ours_graded_condition_counts.csv"), index=False)
condition_seed_cells.to_csv(os.path.join(WORK_DIR, "ours_graded_condition_counts_by_seed.csv"), index=False)
condition_runs.to_csv(os.path.join(WORK_DIR, "ours_graded_condition_count_runs.csv"), index=False)

# Display condition counts 0 through 10 on both axes.
pos_levels = range(11)
neg_levels = range(11)
for modality, label in (("text", "Text"), ("multimodal", "Image")):
    fig, axes = plt.subplots(2, 2, figsize=(13, 11), layout="constrained")
    for ax, family in zip(axes.flat, FAMILY_ORDER):
        sub = condition_cells[(condition_cells["modality"] == modality)
                              & (condition_cells["family"] == family)]
        values = sub.pivot(index="n_negative", columns="n_positive", values="recall").reindex(
            index=neg_levels, columns=pos_levels)
        counts = sub.pivot(index="n_negative", columns="n_positive", values="n_queries").reindex(
            index=neg_levels, columns=pos_levels)
        annotations = values.copy().astype(object)
        for neg in neg_levels:
            for pos in pos_levels:
                value = values.loc[neg, pos]
                annotations.loc[neg, pos] = ("" if pd.isna(value)
                                             else f"{value:.2f}\nn={int(counts.loc[neg, pos])}")
        ax.set_facecolor("0.9")
        sns.heatmap(values, ax=ax, mask=values.isna(), annot=annotations, fmt="",
                    cmap="viridis", vmin=0, vmax=1, square=True, linewidths=0.4,
                    annot_kws={"fontsize": 7}, cbar=False)
        ax.invert_yaxis()
        seeds = sorted(condition_runs.loc[(condition_runs["modality"] == modality)
                                          & (condition_runs["family"] == family), "seed"])
        ax.set_title(f"{TEX_FAMILY[family]} · seeds {', '.join(map(str, seeds))}")
        ax.set_xlabel("Number of positive conditions")
        ax.set_ylabel("Number of negative conditions")
        ax.tick_params(axis="both", labelrotation=0)
    fig.colorbar(axes.flat[0].collections[0], ax=axes.ravel().tolist(), shrink=0.8,
                 label=f"Mean Recall@{k_for(modality)} across seeds")
    fig.suptitle(f"{label} · ours-graded · rephrased test queries\n"
                 "Cell: mean recall and number of queries; grey: no queries", fontsize=14)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(WORK_DIR, f"ours_graded_condition_counts_{modality}.{ext}"), dpi=160)
    plt.show()


### 6b. InfoNCE ours and baseline recall

Uses the Figure 2 models on rephrased test queries, averaging all available trials.
The controls must score exactly the same queries and positives as the graded models;
condition counts are reused from section 8. Both axes span 0–10 and all panels share a
0–1 recall scale. The paper figure is drawn in section 6e.


In [ ]:
# InfoNCE controls from the same Figure 2 selection as the graded heatmaps above.
paper_condition_parts = [condition_query_scores[condition_query_scores["family"] == "infonce"]
                         .assign(role="ours-graded")]
for modality in ("text", "multimodal"):
    ours_runs = condition_runs[(condition_runs["modality"] == modality)
                               & (condition_runs["family"] == "infonce")].sort_values("seed")
    reference_run = ours_runs.iloc[0]
    with open(os.path.join(reference_run.run_dir, "preds", "queries.jsonl")) as handle:
        reference_queries = [json.loads(line) for line in handle if line.strip()]
    reference = {q["query_id"]: (q["query"], tuple(sorted(q["positive_corpus_ids"])))
                 for q in reference_queries}
    counts = condition_query_scores[(condition_query_scores["modality"] == modality)
                                     & (condition_query_scores["family"] == "infonce")
                                     & (condition_query_scores["seed"] == reference_run.seed)]
    counts = counts.set_index("query_id")[["n_positive", "n_negative"]]
    trials = role_trials(PROTOCOL["infonce"], "baseline", modality, CONDITION_KIND)
    if (trials is None or trials["seed"].duplicated().any()
            or set(trials["seed"]) != set(ours_runs["seed"])):
        raise ValueError(f"{modality}: baseline-control trials must match the graded trials")
    k = k_for(modality)
    for run in trials.sort_values("seed").itertuples():
        preds = os.path.join(run.run_dir, "preds")
        with open(os.path.join(preds, "meta.json")) as handle:
            meta = json.load(handle)
        if meta["split"] != "test" or meta["args"]["query_kind"] != ANALYSIS_QUERY_KIND:
            raise ValueError(f"{run.run_dir}: expected rephrased test predictions")
        with open(os.path.join(preds, "queries.jsonl")) as handle:
            queries = [json.loads(line) for line in handle if line.strip()]
        signature = {q["query_id"]: (q["query"], tuple(sorted(q["positive_corpus_ids"])))
                     for q in queries}
        if signature != reference or len(signature) != len(queries) or len(queries) != meta["n_queries"]:
            raise ValueError(f"{run.run_dir}: control and graded test queries do not align")
        rows = []
        for q in queries:
            positives = set(q["positive_corpus_ids"])
            if not positives or len(q["top_k"]) < k:
                raise ValueError(f"{run.run_dir}: incomplete retrieval predictions")
            hits = positives.intersection(int(cid) for cid, _ in q["top_k"][:k])
            rows.append({"query_id": q["query_id"], "recall": len(hits) / len(positives)})
        frame = pd.DataFrame(rows).join(counts, on="query_id", validate="one_to_one")
        np.testing.assert_allclose(frame["recall"].mean(), meta["metrics"][f"recall@{k}"], atol=1e-12)
        paper_condition_parts.append(frame.assign(modality=modality, family="infonce", style=run.style,
                                                  role="baseline", seed=run.seed, k=k))

paper_condition_queries = pd.concat(paper_condition_parts, ignore_index=True)
paper_condition_keys = ["modality", "role", "k", "n_positive", "n_negative"]
paper_condition_by_seed = (paper_condition_queries.groupby(paper_condition_keys + ["seed"], as_index=False)
                           .agg(recall=("recall", "mean"), n_queries=("query_id", "size")))
paper_condition_cells = (paper_condition_by_seed.groupby(paper_condition_keys, as_index=False)
                        .agg(recall=("recall", "mean"), n_queries=("n_queries", "first")))
paper_condition_by_seed.to_csv(os.path.join(WORK_DIR, "infonce_condition_counts_by_seed.csv"), index=False)
paper_condition_cells.to_csv(os.path.join(WORK_DIR, "infonce_condition_counts.csv"), index=False)

STRATEGY_LABEL = {"baseline": "Baseline", "ours-graded": "GOLD mining + reweighting\n(ours)"}


def draw_condition_panels(panels, headings, colorbars, stem):
    """1xN row of 0-10 condition-count heatmaps; writes `FIG_DIR/<stem>.{pdf,png}`.
    `panels` holds one (cells, value, modality, role, xlabel, scale) per axis, where `scale`
    indexes `colorbars`, which hold (vmin, vmax, label, ticks). `headings` holds
    (first, last, text): text centered over axes first..last."""
    with sns.axes_style("white"), plt.rc_context({"font.size": 8, "axes.labelsize": 8,
                                                 "xtick.labelsize": 7, "ytick.labelsize": 7,
                                                 "pdf.fonttype": 42}):
        fig, axes = plt.subplots(1, len(panels), figsize=(7.2, 2.45), sharey=True, layout="constrained")
        for index, (ax, (cells, value, modality, role, xlabel, scale)) in enumerate(zip(axes, panels)):
            vmin, vmax = colorbars[scale][:2]
            sub = cells[(cells["modality"] == modality) & (cells["role"] == role)]
            values = sub.pivot(index="n_negative", columns="n_positive", values=value).reindex(
                index=range(11), columns=range(11))
            ax.set_facecolor("0.9")
            sns.heatmap(values, ax=ax, mask=values.isna(), cmap="viridis", vmin=vmin, vmax=vmax,
                        square=True, linewidths=0, cbar=False, annot=False)
            ax.set_ylim(0, 11)
            ax.set_xticks(np.arange(0, 11, 2) + 0.5, labels=range(0, 11, 2))
            ax.set_yticks(np.arange(0, 11, 2) + 0.5, labels=range(0, 11, 2))
            ax.tick_params(axis="both", labelrotation=0, length=2, pad=2)
            ax.set_xlabel(xlabel, fontsize=7.5, labelpad=3)
            ax.set_ylabel("# of Negative Conditions" if index == 0 else "")
        fig.supxlabel("# of Positive Conditions", fontsize=8)
        for scale, (vmin, vmax, label, ticks) in enumerate(colorbars):
            members = [ax for ax, panel in zip(axes, panels) if panel[-1] == scale]
            fig.colorbar(members[0].collections[0], ax=members, ticks=ticks, fraction=0.025, pad=0.02,
                         label=label)
        # Reserve heading space, then center each heading over its axes.
        fig.get_layout_engine().set(rect=(0, 0, 1, 0.90))
        fig.canvas.draw()
        for first, last, heading in headings:
            left, right = axes[first].get_position(), axes[last].get_position()
            fig.text((left.x0 + right.x1) / 2, max(left.y1, right.y1) + 0.035, heading,
                     ha="center", va="bottom", fontsize=8)
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIG_DIR, f"{stem}.{ext}"), dpi=300, bbox_inches="tight",
                        pad_inches=0.02)
        plt.show()
    return fig, axes


### 6c. Recall difference: ours minus baseline

Subtract the baseline-negative baseline heatmap from the ours heatmap in section 6b,
separately for text and image. These are the same rephrased test queries and model trials;
text uses Recall@5 and images Recall@20. Each cell is **ours − baseline** after averaging
trials: positive values favour ours, negative values favour the baseline, and 0.01 is one
percentage point of recall. Both axes show 0–10. Grey cells have no queries; the shared
diverging colour scale is symmetric about zero.

Writes `analysis/figs/infonce_condition_count_difference.{png,pdf,csv}`.


In [ ]:
# Subtract the aligned, trial-averaged heatmaps from section 8b.
difference_keys = ["modality", "k", "n_positive", "n_negative"]
difference_ours = paper_condition_cells[paper_condition_cells["role"] == "ours-graded"].drop(columns="role")
difference_baseline = paper_condition_cells[paper_condition_cells["role"] == "baseline"].drop(columns="role")
condition_difference = difference_ours.merge(
    difference_baseline, on=difference_keys, how="outer", suffixes=("_ours", "_baseline"),
    validate="one_to_one", indicator=True)
if (condition_difference.empty or (condition_difference["_merge"] != "both").any()
        or (condition_difference["n_queries_ours"] != condition_difference["n_queries_baseline"]).any()):
    raise ValueError("Ours and baseline must have identical condition bins and query counts")
condition_difference["recall_difference"] = (condition_difference["recall_ours"]
                                             - condition_difference["recall_baseline"])
condition_difference = (condition_difference.drop(columns=["_merge", "n_queries_baseline"])
                        .rename(columns={"n_queries_ours": "n_queries"}))
condition_difference.to_csv(os.path.join(WORK_DIR, "infonce_condition_count_difference.csv"), index=False)

# Missing bins stay masked. The shared symmetric scale uses only the displayed 0–10 bins.
difference_tables = {
    modality: condition_difference[condition_difference["modality"] == modality]
    .pivot(index="n_negative", columns="n_positive", values="recall_difference")
    .reindex(index=range(11), columns=range(11))
    for modality in ("text", "multimodal")
}
difference_limit = max(float(np.nanmax(np.abs(table.to_numpy()))) for table in difference_tables.values())
difference_limit = difference_limit or 0.01  # keep a valid scale if every difference is zero
with sns.axes_style("white"):
    difference_fig, difference_axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True,
                                                   layout="constrained")
    for ax, (modality, label) in zip(difference_axes, (("text", "Text"), ("multimodal", "Image"))):
        values = difference_tables[modality]
        ax.set_facecolor("0.9")
        sns.heatmap(values, ax=ax, mask=values.isna(), cmap="RdBu", center=0,
                    vmin=-difference_limit, vmax=difference_limit, square=True,
                    annot=True, fmt="+.2f", annot_kws={"fontsize": 7},
                    linewidths=0.3, cbar=False)
        ax.set_ylim(0, 11)
        ax.set_title(label)
        ax.set_xlabel("Positive conditions")
        ax.set_ylabel("Negative conditions" if modality == "text" else "")
        ax.tick_params(axis="both", labelrotation=0)
    difference_fig.colorbar(difference_axes[0].collections[0], ax=difference_axes,
                             shrink=0.85, label="Recall difference (ours − baseline)")
    for ext in ("png", "pdf"):
        difference_fig.savefig(os.path.join(WORK_DIR, f"infonce_condition_count_difference.{ext}"),
                                dpi=200, bbox_inches="tight")
    plt.show()


### 6d. Hard-negative win rate by positive and negative condition counts

The section 6b panels with pairwise win rate in place of recall: the same InfoNCE
GOLD mining + reweighting and Baseline trials, the same rephrased test queries, and the
same condition bins. Each test query has exactly one verified hard negative; a win is
`sim_pos > sim_neg` on that pair, read from `preds/triplets.jsonl` as in section 4, with
ties counted as losses and easy-negative rows excluded. Cells average queries within a
seed and then seeds with equal weight; every bin must hold the same queries as its
section 6b recall bin. The colour scale runs from chance (0.5) to 1, as in the section 4
bars, so bins below chance clip to the darkest colour; their count is printed.
Writes `paper/figs/infonce_condition_counts_winrate.{pdf,png}` and
`analysis/figs/infonce_condition_counts_winrate{,_by_seed}.csv`.


In [ ]:
# Hard-negative win rate on the section 6b InfoNCE runs, keyed to the same query_ids and
# condition counts. `win_rate` (section 4) scores the whole run; here the same rule is applied
# per query so it can be binned.
winrate_condition_parts = []
for modality in ("text", "multimodal"):
    ours_runs = condition_runs[(condition_runs["modality"] == modality)
                               & (condition_runs["family"] == "infonce")]
    baseline_runs = role_trials(PROTOCOL["infonce"], "baseline", modality, CONDITION_KIND)
    for role, runs in (("ours-graded", ours_runs), ("baseline", baseline_runs)):
        for run in runs.sort_values("seed").itertuples():
            counts = paper_condition_queries[(paper_condition_queries["modality"] == modality)
                                             & (paper_condition_queries["role"] == role)
                                             & (paper_condition_queries["seed"] == run.seed)]
            counts = counts.set_index("query_id")[["n_positive", "n_negative"]]
            with open(os.path.join(run.run_dir, "preds", "triplets.jsonl"), encoding="utf-8") as handle:
                triplets = [json.loads(line) for line in handle if line.strip()]
            hard = pd.DataFrame([t for t in triplets
                                 if t["negative_example_source"] != EASY_NEGATIVE_SOURCE])
            if hard["query_id"].duplicated().any() or set(hard["query_id"]) != set(counts.index):
                raise ValueError(f"{run.run_dir}: expected one hard negative per scored test query")
            frame = pd.DataFrame({"query_id": hard["query_id"],
                                  "win": (hard["sim_pos"] > hard["sim_neg"]).astype(float)})
            frame = frame.join(counts, on="query_id", validate="one_to_one")
            np.testing.assert_allclose(frame["win"].mean(), win_rate(run.run_dir)["win_rate"], atol=1e-12)
            winrate_condition_parts.append(frame.assign(modality=modality, role=role, seed=run.seed))

winrate_condition_queries = pd.concat(winrate_condition_parts, ignore_index=True)
winrate_keys = ["modality", "role", "n_positive", "n_negative"]
winrate_condition_by_seed = (winrate_condition_queries.groupby(winrate_keys + ["seed"], as_index=False)
                             .agg(win_rate=("win", "mean"), n_queries=("query_id", "size")))
winrate_condition_cells = (winrate_condition_by_seed.groupby(winrate_keys, as_index=False)
                           .agg(win_rate=("win_rate", "mean"), n_queries=("n_queries", "first")))
aligned = winrate_condition_cells.merge(paper_condition_cells[winrate_keys + ["n_queries"]],
                                        on=winrate_keys, how="outer", validate="one_to_one",
                                        suffixes=("", "_recall"), indicator=True)
if (aligned["_merge"] != "both").any() or (aligned["n_queries"] != aligned["n_queries_recall"]).any():
    raise ValueError("Win-rate bins must hold the same queries as the section 6b recall bins")
winrate_condition_by_seed.to_csv(os.path.join(WORK_DIR, "infonce_condition_counts_winrate_by_seed.csv"), index=False)
winrate_condition_cells.to_csv(os.path.join(WORK_DIR, "infonce_condition_counts_winrate.csv"), index=False)

shown = winrate_condition_cells[(winrate_condition_cells["n_positive"] <= 10)
                                & (winrate_condition_cells["n_negative"] <= 10)]
print(f"{int((shown['win_rate'] < 0.5).sum())} of {len(shown)} displayed bins are below chance "
      "and clip to the bottom of the colour scale")
display(winrate_condition_cells.groupby(["modality", "role"])["win_rate"].describe().round(3))
winrate_condition_fig, winrate_condition_axes = draw_condition_panels(
    [(winrate_condition_cells, "win_rate", modality, role, STRATEGY_LABEL[role], 0)
     for modality in ("text", "multimodal") for role in ("baseline", "ours-graded")],
    [(0, 1, "Text win rate"), (2, 3, "Image win rate")],
    [(0.5, 1, "Win rate", [0.5, 0.75, 1])], "infonce_condition_counts_winrate")


### 6e. Paper figure: GOLD mining + reweighting recall and win rate

A 1×4 row for the results section: InfoNCE GOLD mining + reweighting text recall, image recall,
text win rate, and image win rate, from the section 6b and 6d cells. Recall uses a 0–1 scale and
win rate the section 6d 0.5–1 scale. Writes `paper/figs/infonce_condition_counts.{pdf,png}`.

In [ ]:
paper_condition_fig, paper_condition_axes = draw_condition_panels(
    [(paper_condition_cells, "recall", "text", "ours-graded", "", 0),
     (paper_condition_cells, "recall", "multimodal", "ours-graded", "", 0),
     (winrate_condition_cells, "win_rate", "text", "ours-graded", "", 1),
     (winrate_condition_cells, "win_rate", "multimodal", "ours-graded", "", 1)],
    [(0, 0, "Text Recall@5"), (1, 1, "Image Recall@20"), (2, 2, "Text win rate"), (3, 3, "Image win rate")],
    [(0, 1, "Recall", [0, 0.5, 1]), (0.5, 1, "Win rate", [0.5, 0.75, 1])],
    "infonce_condition_counts")

## 7. Collected and sampled human-study attribute distributions

The combined datasets contain every usable annotation from the freshly downloaded first study and the
[text backfill](https://docs.google.com/spreadsheets/d/1aT9mDmMGltYGf4avmDjO8NwIsVR7d6C4fXBEJwT-8kA/edit)
and [image backfill](https://docs.google.com/spreadsheets/d/1ZOtTNWVY92cwqFhX7IY-hvS8y6eiK32-aXPVKFCfDYs/edit).
The image tab `Lorena fixed` supersedes `Lorena`. Blank or flagged answers and pairs absent
from the processed base are excluded, following the original importer.

The sampled curve reduces overrepresented low-attribute counts. Separately for text and
image, let h(k) be the number of human queries with k total stated attributes. Set the cap
M = max h(k) over k >= 6, and retain min(h(k), M) queries from each count k. Sampling within
overfull bins is uniform without replacement with seed 42; all bins at or below the cap
are preserved. Every query with 6+ attributes is retained. This limits tall bins without
filling gaps or forcing an exact match to the synthetic histogram. The figure overlays
synthetic, first-study, full combined, and sampled human distributions.

Run All starts with `.venv/bin/python analysis/refresh_human.py`, which downloads all four
spreadsheets, rebuilds these datasets, and refreshes changed human predictions.
This section runs `paper/draw_attr_hist.py`, which refreshes the sampled datasets through
`human_study/sample_human_labels.py` and writes `tmp/attr_hist.{png,pdf,json,csv}`.
Samples are saved as `_human-matched`, with source indices, bin counts, cap, and
seed in `sampling_report.json`; the full combined datasets remain available.

Synthetic data uses distinct current rephrased-in-context test queries; human counts come
from the Q1/Q2 lists. Synthetic text counts use the feature lists recorded during BM25 generation;
image counts use stored feature lists. Commas inside features do not change the counts.

`_human-matched-in-context` removes any of the five fixed style examples that
survive sampling. All human results above use this refreshed matched evaluation set. Table 2 excludes
the same fixed-example products as Figure 2a.


In [ ]:
import json
from pathlib import Path
import subprocess

import pandas as pd
from IPython.display import Image, display

attribute_root = Path.cwd()
attribute_run = subprocess.run(
    [str(attribute_root / '.venv/bin/python'), '-B', 'paper/draw_attr_hist.py'],
    cwd=attribute_root, check=True, capture_output=True, text=True,
)
print(attribute_run.stdout)
attribute_report = json.loads((attribute_root / 'tmp/attr_hist.json').read_text())
attribute_table = pd.DataFrame([
    {'modality': modality, 'dataset': dataset, **stats}
    for modality, report in attribute_report.items()
    for dataset, stats in report['statistics'].items()
]).set_index(['modality', 'dataset'])
display(attribute_table.round(2))
attribute_sampling = pd.DataFrame([
    {'modality': modality, **{key: report['sampling'][key] for key in
        ['seed', 'cap_per_attribute_count', 'source_rows', 'sampled_rows',
         'removed_rows', 'sampled_in_context_rows']}}
    for modality, report in attribute_report.items()
]).set_index('modality')
display(attribute_sampling)
display(Image(filename=str(attribute_root / 'tmp/attr_hist.png')))
